In [ ]:
# set_sequence_classifier.py

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split, TimeSeriesSplit
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix
from tqdm import tqdm
import warnings

warnings.filterwarnings("ignore")

# ==============================================================================
# 1. Configuration Dictionary
# ==============================================================================
# Centralized configuration for all hyperparameters and settings.
config = {
    # --- Data Configuration ---
    "data": {
        "test_size": 0.2,
        "sequence_length": 50,
        "target_col": "target",
        "unit_id_col": "unit_id",
        "time_col": "time_period"
    },
    # --- Model Architecture ---
    "model": {
        "d_model": 128,
        "d_set_summary": 16,
        "phi_hidden_dim": 256,
        "rho_hidden_dim": 64,
        "psi_hidden_dim": 128,
        "n_set_seq_layers": 4,
        "n_seq_layers": 2,
        "dropout": 0.1,
        "long_conv_kernel_size": 32,
    },
    # --- Training Configuration ---
    "training": {
        "loss_function": "gmean", # Options: "bce", "gmean"
        "learning_rate": 0.001,
        "batch_size": 32, # Reduced batch size to accommodate larger tensors
        "n_epochs": 20,
        "device": "cuda" if torch.cuda.is_available() else "cpu",
        "scaler": "standard",
    },
    # --- Cross-Validation ---
    "cross_val": {
        "n_splits": 5
    }
}

# ==============================================================================
# 2. Model Components & Architecture
# ==============================================================================

class GMeanLoss(nn.Module):
    """
    A differentiable loss function to maximize the G-mean.
    Loss = 1 - G-mean = 1 - sqrt(Sensitivity * Specificity).
    """
    def __init__(self, epsilon=1e-8):
        super().__init__()
        self.epsilon = epsilon

    def forward(self, logits, labels):
        preds = torch.sigmoid(logits)
        labels = labels.float()

        tp = torch.sum(preds * labels)
        sensitivity = tp / (torch.sum(labels) + self.epsilon)
        specificity = torch.sum((1 - preds) * (1 - labels)) / (torch.sum(1 - labels) + self.epsilon)

        g_mean = torch.sqrt(sensitivity * specificity + self.epsilon)
        return 1 - g_mean

class LongConv(nn.Module):
    """A simple 1D causal convolution layer."""
    def __init__(self, d_model, kernel_size, dropout):
        super().__init__()
        self.conv = nn.Conv1d(
            in_channels=d_model,
            out_channels=d_model,
            kernel_size=kernel_size,
            padding=kernel_size - 1,
            groups=d_model
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = x.transpose(1, 2)
        x = self.conv(x)
        x = x[:, :, :-(self.conv.kernel_size[0] - 1)]
        x = x.transpose(1, 2)
        return self.dropout(x)


class SetSequenceLayer(nn.Module):
    """Implements a single Set-Sequence layer."""
    def __init__(self, config):
        super().__init__()
        d_model = config["model"]["d_model"]
        d_set_summary = config["model"]["d_set_summary"]
        phi_hidden = config["model"]["phi_hidden_dim"]
        rho_hidden = config["model"]["rho_hidden_dim"]
        psi_hidden = config["model"]["psi_hidden_dim"]
        kernel_size = config["model"]["long_conv_kernel_size"]
        dropout = config["model"]["dropout"]

        self.phi = nn.Sequential(nn.Linear(d_model, phi_hidden), nn.ReLU(), nn.Linear(phi_hidden, d_model))
        self.rho = nn.Sequential(nn.Linear(d_model, rho_hidden), nn.ReLU(), nn.Linear(rho_hidden, d_set_summary))
        self.psi = nn.Sequential(nn.Linear(d_model + d_set_summary, psi_hidden), nn.ReLU(), nn.Linear(psi_hidden, d_model))
        self.seq_layer = LongConv(d_model, kernel_size, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):
        batch_size, num_units, seq_len, d_model = x.shape

        x_reshaped = x.view(batch_size * num_units, seq_len, d_model)
        phi_x = self.phi(x_reshaped).view(batch_size, num_units, seq_len, d_model)
        mean_phi_x = torch.mean(phi_x, dim=1)
        set_summary = self.rho(mean_phi_x)

        set_summary_expanded = set_summary.unsqueeze(1).expand(-1, num_units, -1, -1)
        augmented_x = torch.cat([x, set_summary_expanded], dim=-1)
        augmented_x_reshaped = augmented_x.view(batch_size * num_units, seq_len, -1)
        psi_out = self.psi(augmented_x_reshaped)

        res_x = x.view(batch_size * num_units, seq_len, d_model)
        processed_x = self.norm1(res_x + psi_out)
        seq_out = self.seq_layer(processed_x)
        final_out = self.norm2(processed_x + seq_out)

        return final_out.view(batch_size, num_units, seq_len, d_model)


class SetSequenceClassifier(nn.Module):
    """The full Set-Sequence model for per-unit classification."""
    def __init__(self, config, n_features):
        super().__init__()
        d_model = config["model"]["d_model"]
        n_set_seq_layers = config["model"]["n_set_seq_layers"]
        n_seq_layers = config["model"]["n_seq_layers"]
        kernel_size = config["model"]["long_conv_kernel_size"]
        dropout = config["model"]["dropout"]

        self.input_projection = nn.Linear(n_features, d_model)
        self.set_seq_layers = nn.ModuleList([SetSequenceLayer(config) for _ in range(n_set_seq_layers)])
        self.final_seq_layers = nn.ModuleList([LongConv(d_model, kernel_size, dropout) for _ in range(n_seq_layers)])
        self.classifier_head = nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model, 1))

    def forward(self, x):
        """
        Args:
            x (torch.Tensor): Input of shape (batch_size, num_units, seq_len, n_features)
        Returns:
            torch.Tensor: Logits of shape (batch_size, num_units)
        """
        batch_size, num_units, seq_len, _ = x.shape

        # 1. Project input features to d_model
        x = self.input_projection(x)

        # 2. Pass through Set-Sequence layers
        for layer in self.set_seq_layers:
            x = layer(x)

        # 3. Reshape to process all units' sequences in one go
        # (batch * units, seq_len, d_model)
        x = x.view(batch_size * num_units, seq_len, -1)

        # 4. Pass through final sequence-only layers
        for layer in self.final_seq_layers:
            x = x + layer(x)

        # 5. Use the last time step's output for classification
        last_time_step = x[:, -1, :]

        # 6. Get logits from the classifier head
        logits = self.classifier_head(last_time_step)

        # 7. Reshape back to (batch_size, num_units)
        return logits.view(batch_size, num_units)

# ==============================================================================
# 3. Data Preparation and Utilities
# ==============================================================================

def create_sequences(df, config):
    """
    Transforms the DataFrame into sequences and per-unit targets.
    X shape: (num_sequences, num_units, seq_len, n_features)
    y shape: (num_sequences, num_units)
    """
    seq_len = config["data"]["sequence_length"]
    unit_col = config["data"]["unit_id_col"]
    time_col = config["data"]["time_col"]
    target_col = config["data"]["target_col"]

    feature_cols = [c for c in df.columns if c not in [unit_col, time_col, target_col]]
    n_features = len(feature_cols)

    df = df.sort_values(by=[time_col, unit_col])

    sequences, targets = [], []
    unique_times = df[time_col].unique()

    for t in range(seq_len, len(unique_times)):
        start_time, end_time, target_time = unique_times[t - seq_len], unique_times[t - 1], unique_times[t]

        sequence_df = df[(df[time_col] >= start_time) & (df[time_col] <= end_time)]
        target_df = df[df[time_col] == target_time]

        # Get the units present in this time window
        units_in_window = sequence_df[unit_col].unique()

        # Pivot features
        seq_pivot = sequence_df.pivot(index=unit_col, columns=time_col, values=feature_cols).fillna(0)

        # Pivot targets
        target_pivot = target_df.pivot(index=unit_col, columns=time_col, values=target_col)

        # Align units between features and targets, fill missing targets with 0
        seq_pivot, target_pivot = seq_pivot.align(target_pivot, join='left', axis=0, fill_value=0)

        num_units = len(seq_pivot)
        if num_units == 0: continue

        try:
            seq_array = seq_pivot.values.reshape(num_units, seq_len, n_features)
            target_array = target_pivot.values.flatten()

            sequences.append(seq_array)
            targets.append(target_array)
        except ValueError:
            continue

    # Note: Sequences can have different numbers of units. This requires custom padding/batching.
    # For simplicity here, we filter for sequences with the most common number of units.
    if not sequences: return np.array([]), np.array([])

    unit_counts = [s.shape[0] for s in sequences]
    if not unit_counts: return np.array([]), np.array([])

    most_common_n_units = max(set(unit_counts), key=unit_counts.count)

    X_filtered = [sequences[i] for i, count in enumerate(unit_counts) if count == most_common_n_units]
    y_filtered = [targets[i] for i, count in enumerate(unit_counts) if count == most_common_n_units]

    if not X_filtered: return np.array([]), np.array([])

    return np.stack(X_filtered), np.stack(y_filtered)


def get_scaler(name):
    if name == "standard": return StandardScaler()
    if name == "minmax": return MinMaxScaler()
    return None

def get_criterion(name):
    if name == "bce": return nn.BCEWithLogitsLoss()
    if name == "gmean": return GMeanLoss()
    raise ValueError(f"Unknown loss function: {name}")

def calculate_gmean(labels, preds, epsilon=1e-8):
    if len(np.unique(labels)) < 2: return 0.0
    cm = confusion_matrix(labels, np.round(preds))
    if cm.shape != (2, 2): return 0.0
    tn, fp, fn, tp = cm.ravel()
    sensitivity = tp / (tp + fn + epsilon)
    specificity = tn / (tn + fp + epsilon)
    return np.sqrt(sensitivity * specificity)

# ==============================================================================
# 4. Training and Evaluation Loop
# ==============================================================================

def train_epoch(model, dataloader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for X_batch, y_batch in dataloader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        outputs = model(X_batch) # Shape: (batch, units)

        # Flatten outputs and targets for loss calculation
        loss = criterion(outputs.view(-1), y_batch.view(-1).float())

        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(dataloader)

def evaluate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for X_batch, y_batch in dataloader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = model(X_batch)

            # Flatten for loss and metrics
            flat_outputs = outputs.view(-1)
            flat_labels = y_batch.view(-1).float()

            loss = criterion(flat_outputs, flat_labels)
            total_loss += loss.item()

            all_preds.extend(torch.sigmoid(flat_outputs).cpu().numpy())
            all_labels.extend(flat_labels.cpu().numpy())

    avg_loss = total_loss / len(dataloader)
    auc = roc_auc_score(all_labels, all_preds)
    acc = accuracy_score(all_labels, np.round(all_preds))
    gmean = calculate_gmean(all_labels, all_preds)

    return avg_loss, auc, acc, gmean

# ==============================================================================
# 5. Main Execution
# ==============================================================================

def main():
    print("--- Set-Sequence Model for Per-Unit Classification ---")
    print(f"Using device: {config['training']['device']}")
    print(f"Optimizing with loss function: {config['training']['loss_function'].upper()}")

    print("Generating sample data...")
    n_units, n_time_periods, n_features = 50, 500, 10
    data = []
    for unit in range(n_units):
        for time in range(n_time_periods):
            row = {'unit_id': unit, 'time_period': time}
            features = np.sin(time / 50 + unit) + np.random.randn(n_features) * 0.5
            for i, f_val in enumerate(features): row[f'feature_{i}'] = f_val
            row['target'] = 1 if (features[0] + np.sin(time/20)) > 0.8 else 0
            data.append(row)
    df = pd.DataFrame(data)
    print(f"Sample data created. Target distribution:\n{df['target'].value_counts(normalize=True)}")

    feature_cols = [c for c in df.columns if isinstance(c, str) and c.startswith('feature')]
    time_col = config['data']['time_col']
    test_split_time = df[time_col].unique()[int(len(df[time_col].unique()) * (1 - config['data']['test_size']))]
    df_train_val, df_test = df[df[time_col] < test_split_time], df[df[time_col] >= test_split_time]

    scaler = get_scaler(config['training']['scaler'])
    if scaler:
        print(f"Applying {config['training']['scaler']} scaling...")
        df_train_val.loc[:, feature_cols] = scaler.fit_transform(df_train_val[feature_cols])
        df_test.loc[:, feature_cols] = scaler.transform(df_test[feature_cols])

    print("\nStarting walk-forward cross-validation...")
    tscv = TimeSeriesSplit(n_splits=config['cross_val']['n_splits'])
    fold_results = []
    time_periods = df_train_val[time_col].unique()

    for fold, (train_indices, val_indices) in enumerate(tscv.split(time_periods)):
        print(f"\n--- Fold {fold + 1}/{config['cross_val']['n_splits']} ---")
        df_train_fold = df_train_val[df_train_val[time_col].isin(time_periods[train_indices])]
        df_val_fold = df_train_val[df_train_val[time_col].isin(time_periods[val_indices])]

        X_train, y_train = create_sequences(df_train_fold, config)
        X_val, y_val = create_sequences(df_val_fold, config)

        if X_train.shape[0] == 0 or X_val.shape[0] == 0:
            print("Skipping fold due to insufficient data to create sequences.")
            continue

        train_loader = DataLoader(TensorDataset(torch.from_numpy(X_train).float(), torch.from_numpy(y_train).long()), batch_size=config['training']['batch_size'], shuffle=True)
        val_loader = DataLoader(TensorDataset(torch.from_numpy(X_val).float(), torch.from_numpy(y_val).long()), batch_size=config['training']['batch_size'])

        model = SetSequenceClassifier(config, n_features=len(feature_cols)).to(config['training']['device'])
        optimizer = optim.Adam(model.parameters(), lr=config['training']['learning_rate'])
        criterion = get_criterion(config['training']['loss_function'])

        for epoch in range(config['training']['n_epochs']):
            train_loss = train_epoch(model, train_loader, optimizer, criterion, config['training']['device'])
            val_loss, val_auc, val_acc, val_gmean = evaluate(model, val_loader, criterion, config['training']['device'])
            if (epoch + 1) % 5 == 0:
                 print(f"Epoch {epoch+1:02d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val AUC: {val_auc:.4f} | Val G-mean: {val_gmean:.4f}")

        fold_results.append({'auc': val_auc, 'acc': val_acc, 'gmean': val_gmean})

    print("\n--- Final Evaluation on Test Set ---")
    print("Retraining model on full train/val data...")
    X_train_full, y_train_full = create_sequences(df_train_val, config)
    X_test, y_test = create_sequences(df_test, config)

    if X_train_full.shape[0] == 0 or X_test.shape[0] == 0:
        print("Cannot perform final evaluation due to insufficient data.")
        return

    train_full_loader = DataLoader(TensorDataset(torch.from_numpy(X_train_full).float(), torch.from_numpy(y_train_full).long()), batch_size=config['training']['batch_size'], shuffle=True)
    test_loader = DataLoader(TensorDataset(torch.from_numpy(X_test).float(), torch.from_numpy(y_test).long()), batch_size=config['training']['batch_size'])

    final_model = SetSequenceClassifier(config, n_features=len(feature_cols)).to(config['training']['device'])
    optimizer = optim.Adam(final_model.parameters(), lr=config['training']['learning_rate'])
    criterion = get_criterion(config['training']['loss_function'])

    for epoch in range(config['training']['n_epochs']):
        train_loss = train_epoch(final_model, train_full_loader, optimizer, criterion, config['training']['device'])
        if (epoch + 1) % 5 == 0: print(f"Retraining Epoch {epoch+1:02d} | Train Loss: {train_loss:.4f}")

    test_loss, test_auc, test_acc, test_gmean = evaluate(final_model, test_loader, criterion, config['training']['device'])

    print("\n--- Results Summary ---")
    if fold_results:
        avg_cv_gmean = np.mean([r['gmean'] for r in fold_results])
        print(f"Average Cross-Validation G-mean: {avg_cv_gmean:.4f}")

    print(f"\nFinal Test Set G-mean: {test_gmean:.4f}")
    print(f"Final Test Set AUC: {test_auc:.4f}")
    print(f"Final Test Set Accuracy: {test_acc:.4f}")


if __name__ == "__main__":
    main()

In [ ]:
# -*- coding: utf-8 -*-
# =====================================================================
# Cell: Model Training and Evaluation (LSTM with Optional Gating)
# =====================================================================
# Performs sequence creation, data splitting, scaling, model training
# (with LSTM + optional gating), hyperparameter tuning (optional),
# evaluation, and saves results into a timestamped subfolder.
# Includes a separate function to reload a model and run predictions.

# --- Standard Library Imports ---
import math
import time
import sys
import os
import logging
from datetime import datetime
from functools import partial # For passing args to Optuna objective
import json

# --- Data Handling and Numerical Computation ---
import numpy as np
import pandas as pd

# --- Machine Learning & Deep Learning ---
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau
import optuna
from optuna.trial import TrialState # For callback check
from optuna.exceptions import DuplicatedStudyError # To handle study deletion attempts

# --- Scikit-learn ---
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    f1_score, accuracy_score, roc_auc_score, precision_recall_curve,
    classification_report, confusion_matrix, recall_score # recall_score needed for gmean
)

# --- Imbalanced-learn ---
# NOTE: Undersampling is disabled in this version
try:
    from imblearn.under_sampling import RandomUnderSampler
except ImportError:
    logging.warning("`imbalanced-learn` library not found. Undersampling is disabled anyway.")
    RandomUnderSampler = None

# --- Plotting & Jupyter Integration ---
import matplotlib.pyplot as plt # Still used for final static plot & confusion matrix
import matplotlib.dates as mdates # For formatting dates on plots
import seaborn as sns # Used for confusion matrix
import plotly.graph_objects as go # For live plotting
from plotly.subplots import make_subplots # To create subplots
try:
    from IPython.display import display, Image # To display Plotly widget and saved images in Jupyter
    ipython_display_available = True
except ImportError:
    logging.warning("IPython.display not available. Plots will not be displayed inline.")
    ipython_display_available = False
    # Define dummy functions if display is not available to avoid NameError later
    def display(*args, **kwargs): pass
    def Image(*args, **kwargs): pass
# Note: ipywidgets is used implicitly by FigureWidget

# ========================================================
# Logging Setup
# ========================================================
# Configure logging settings once at the top level
LOG_LEVEL = logging.INFO
LOG_FORMAT = '%(asctime)s - %(levelname)s - %(module)s - %(message)s'
logging.basicConfig(level=LOG_LEVEL, format=LOG_FORMAT, datefmt='%Y-%m-%d %H:%M:%S', force=True)

# ========================================================
# Helper Functions & Classes (Model, Training, Evaluation)
# ========================================================

# --- Reproducibility Helper ---
def set_seed(seed_value):
    """Sets the random seed for reproducibility across libraries."""
    np.random.seed(seed_value)
    torch.manual_seed(seed_value)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed_value)
        torch.cuda.manual_seed_all(seed_value)
        # Ensure deterministic behavior for CuDNN (can impact performance)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    logging.info(f"Random seed set to {seed_value}")

# --- Sequence Creation and Splitting Helpers ---
def create_sequences(input_data, target_data, seq_length):
    """Creates sequences and corresponding labels from time series data."""
    sequences, labels = [], []
    if len(input_data) <= seq_length:
        logging.warning(f"Input data length ({len(input_data)}) <= seq length ({seq_length}). Cannot create sequences.")
        return np.array(sequences), np.array(labels)
    # Ensure numpy arrays
    if isinstance(input_data, pd.DataFrame): input_data = input_data.values
    if isinstance(target_data, pd.Series): target_data = target_data.values

    for i in range(len(input_data) - seq_length):
        sequences.append(input_data[i:i + seq_length+1]) # Sequence includes data up to t-1
        labels.append(target_data[i + seq_length]) # Label is at time t
    return np.array(sequences), np.array(labels)

def log_class_distribution(labels, dataset_name):
    """Logs the class distribution of a label array."""
    if labels is None or len(labels) == 0:
        logging.warning(f"Cannot log class distribution for {dataset_name}: labels are empty or None.")
        return
    try:
        # Ensure labels are integers for unique counts
        unique_classes, counts = np.unique(labels.astype(int), return_counts=True)
        distribution = dict(zip(unique_classes, counts))
        total_samples = len(labels)
        ratios = {k: f"{(v/total_samples)*100:.2f}%" for k, v in distribution.items()}
        logging.info(f"{dataset_name} class distribution - Counts: {distribution}, Ratios: {ratios}")
    except Exception as e:
        logging.error(f"Could not calculate class distribution for {dataset_name}: {e}")

def split_apply_undersample_scale(data_df, config):
    """
    Splits data chronologically, creates sequences, applies scaling based on the training set,
    and returns the processed data along with the original test indices and scaler.
    Uses the SEED from the config for reproducibility.
    """
    logging.info("--- Starting Data Splitting, Sequencing, and Scaling ---")
    # Ensure seed is set for any potential future random operations within this function
    set_seed(config['SEED'])

    if 'target' not in data_df.columns: logging.error("Column 'target' not found."); sys.exit(1)
    features_df = data_df.drop('target', axis=1)
    if features_df.empty: logging.error("No feature columns found."); sys.exit(1)
    target_series = data_df['target']

    X_raw = features_df.values; y_raw = target_series.values
    original_index = data_df.index
    n_features = X_raw.shape[1]
    logging.info(f"Separated raw features ({n_features}) and target.")

    n_samples_raw = len(X_raw)
    train_end_idx = int(n_samples_raw * config['TRAIN_SPLIT_RATIO'])
    val_end_idx = train_end_idx + int(n_samples_raw * config['VALIDATION_SPLIT_RATIO'])

    X_train_raw, y_train_raw = X_raw[:train_end_idx], y_raw[:train_end_idx]
    X_val_raw, y_val_raw = X_raw[train_end_idx:val_end_idx], y_raw[train_end_idx:val_end_idx]
    X_test_raw, y_test_raw = X_raw[val_end_idx:], y_raw[val_end_idx:]
    test_indices_raw = original_index[val_end_idx:]

    logging.info(f"Chronological split: Train={len(X_train_raw)}, Val={len(X_val_raw)}, Test={len(X_test_raw)}")
    log_class_distribution(y_train_raw, "Raw Training Set")
    log_class_distribution(y_val_raw, "Raw Validation Set")
    log_class_distribution(y_test_raw, "Raw Test Set")

    if config['USE_UNDERSAMPLING']: logging.warning("Undersampling enabled but not applied before sequencing.")
    else: logging.info("Undersampling disabled.")
    X_train_processed, y_train_processed = X_train_raw, y_train_raw

    logging.info("Creating sequences...")
    seq_length = config['SEQUENCE_LENGTH']
    X_train_seq, y_train_seq = create_sequences(X_train_processed, y_train_processed, seq_length)
    X_val_seq, y_val_seq = create_sequences(X_val_raw, y_val_raw, seq_length)
    X_test_seq, y_test_seq = create_sequences(X_test_raw, y_test_raw, seq_length)

    # Adjust test indices to align with the sequence labels
    test_indices_seq = None
    if len(test_indices_raw) > seq_length:
        test_indices_seq = test_indices_raw[seq_length:]
        if len(test_indices_seq) != len(y_test_seq):
            logging.error(f"Mismatch between length of sequential test labels ({len(y_test_seq)}) and adjusted test indices ({len(test_indices_seq)}). Check logic.")
            test_indices_seq = None
        else:
            logging.info(f"Aligned test indices with sequence labels. Length: {len(test_indices_seq)}")
    elif len(y_test_seq) > 0: # If sequences were created but too short for index alignment
        logging.warning(f"Raw test set length ({len(test_indices_raw)}) not greater than sequence length ({seq_length}). Cannot align indices for test sequences.")
    # else: y_test_seq is empty, no indices needed

    if X_train_seq.size == 0 or X_val_seq.size == 0 or (X_test_seq.size == 0 and len(y_test_seq) > 0): # Check if test is unexpectedly empty
        logging.error("Sequence creation resulted in empty set(s). Check SEQUENCE_LENGTH vs data sizes."); sys.exit(1)

    logging.info(f"Sequences created. Shapes: Train={X_train_seq.shape}, Val={X_val_seq.shape}, Test={X_test_seq.shape}")
    log_class_distribution(y_train_seq, "Training Sequences (Labels)")
    log_class_distribution(y_val_seq, "Validation Sequences (Labels)")
    log_class_distribution(y_test_seq, "Test Sequences (Labels)")

    logging.info("Fitting StandardScaler on training sequences...")
    if config.get('USE_DUMMY_SCALER', False):
        logging.info("Using dummy scaler. No scaling will be applied.")
        # Define a dummy scaler with identity transforms
        class DummyScaler:
            def fit(self, X):
                return self
            def transform(self, X):
                return X
            def inverse_transform(self, X):
                return X
        scaler = DummyScaler().fit(X_train_seq.reshape(-1, n_features))
        X_train = X_train_seq
        X_val = X_val_seq
        X_test = X_test_seq if X_test_seq.size > 0 else np.array([])
    else:
        scaler = StandardScaler()
        train_shape = X_train_seq.shape
        # Fit ONLY on training data
        scaler.fit(X_train_seq.reshape(-1, n_features))
        logging.info("StandardScaler fitted.")

        logging.info("Applying StandardScaler...")
        # Transform and reshape back to 3D (samples, seq_len, n_features)
        X_train = scaler.transform(X_train_seq.reshape(-1, n_features)).reshape(train_shape)
        X_val = scaler.transform(X_val_seq.reshape(-1, n_features)).reshape(X_val_seq.shape)
        # Handle empty test set case for scaling
        if X_test_seq.size > 0:
            X_test = scaler.transform(X_test_seq.reshape(-1, n_features)).reshape(X_test_seq.shape)
        else:
            X_test = np.array([]) # Keep it as an empty array
            logging.info("Test set is empty, skipping scaling for test set.")

    logging.info("Scaling applied.")
    logging.info("--- Data Splitting, Sequencing, and Scaling Finished ---")
    # Return the scaler object as well
    return X_train, y_train_seq, X_val, y_val_seq, X_test, y_test_seq, n_features, scaler, y_train_raw, test_indices_seq


# --- PyTorch Dataset ---
class TimeSeriesDataset(Dataset):
    """ Custom PyTorch Dataset for time series sequences. """
    def __init__(self, sequences, labels):
        # Convert to tensors if they aren't already
        self.sequences = torch.tensor(sequences, dtype=torch.float32) if not isinstance(sequences, torch.Tensor) else sequences.float()
        self.labels = torch.tensor(labels, dtype=torch.float32) if not isinstance(labels, torch.Tensor) else labels.float()

    def __len__(self): return len(self.sequences)
    def __getitem__(self, idx): return self.sequences[idx], self.labels[idx]


# --- Model Architecture Components (Gating) ---
class DynamicFeatureWeighting(nn.Module):
    """ Applies a learned gating mechanism to modulate feature importance. """

    def __init__(self, n_features: int, dropout_rate: float):
        super().__init__()
        # Simple MLP gate: Linear -> ReLU -> Dropout -> Linear -> Sigmoid

        self.gate_fc1 = nn.Linear(n_features, n_features)
        self.gate_relu = nn.ReLU()
        self.gate_fc2 = nn.Linear(n_features, n_features)
        self.gate_sigmoid = nn.Sigmoid()
        self.dropout = nn.Dropout(dropout_rate)
        # Initialize weights
        nn.init.xavier_uniform_(self.gate_fc1.weight); nn.init.zeros_(self.gate_fc1.bias)
        nn.init.xavier_uniform_(self.gate_fc2.weight); nn.init.zeros_(self.gate_fc2.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:

        gates = self.gate_fc1(x)
        gates = self.gate_relu(gates)
        gates = self.dropout(gates)
        gates = self.gate_fc2(gates)
        gates = self.gate_sigmoid(gates) # Gate values between 0 and 1
        return x * gates # Element-wise multiplication

# --- Model Architecture (LSTM Gated Classifier) ---
class LSTMGatedClassifier(nn.Module):
    """ LSTM model with optional dynamic feature gating for classification. """

    def __init__(self, n_features: int, use_dynamic_weighting: bool,
                 lstm_hidden_size: int, lstm_n_layers: int, lstm_dropout: float, fc_dropout: float, n_classes: int = 1):
        super().__init__()
        self.use_dynamic_weighting = use_dynamic_weighting
        self.lstm_n_layers = lstm_n_layers
        self.lstm_hidden_size = lstm_hidden_size
        self.n_classes = n_classes # Typically 1 for binary classification with BCEWithLogitsLoss

        # Store init args for reloading

        self.init_args = {
            'n_features': n_features,
            'use_dynamic_weighting': use_dynamic_weighting, 'lstm_hidden_size': lstm_hidden_size,
            'lstm_n_layers': lstm_n_layers, 'lstm_dropout': lstm_dropout, 'fc_dropout': fc_dropout,
            'n_classes': n_classes
        }


        # 2. Optional Gating Layer
        # <<< CHANGE: Pass n_features to DynamicFeatureWeighting >>>
        self.dynamic_weighting = DynamicFeatureWeighting(n_features, fc_dropout) if use_dynamic_weighting else None

        # 3. LSTM Layer

        # Apply dropout between LSTM layers only if more than one layer exists
        lstm_input_dropout = lstm_dropout if lstm_n_layers > 1 else 0.0
        self.lstm = nn.LSTM(input_size=n_features, hidden_size=lstm_hidden_size,
                             num_layers=lstm_n_layers, batch_first=True, dropout=lstm_input_dropout)

        # 4. Final Classification Head
        self.dropout = nn.Dropout(fc_dropout)
        self.fc = nn.Linear(lstm_hidden_size, n_classes) # Output logits

        self._init_lstm_weights() # Initialize LSTM and FC weights

    def _init_lstm_weights(self):
        """ Initializes LSTM weights for better stability. """
        for name, param in self.lstm.named_parameters():
            if 'bias' in name: nn.init.constant_(param, 0.0)
            elif 'weight_ih' in name: nn.init.xavier_uniform_(param) # Input-hidden weights
            elif 'weight_hh' in name: nn.init.orthogonal_(param) # Hidden-hidden weights (good for recurrence)
        # Initialize final FC layer
        nn.init.xavier_uniform_(self.fc.weight); nn.init.zeros_(self.fc.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x shape: (batch, seq_len, n_features)


        if self.dynamic_weighting:
            x = self.dynamic_weighting(x) # -> (batch, seq_len, n_features)

        # Initialize LSTM hidden and cell states
        h0 = torch.zeros(self.lstm_n_layers, x.size(0), self.lstm_hidden_size).to(x.device)
        c0 = torch.zeros(self.lstm_n_layers, x.size(0), self.lstm_hidden_size).to(x.device)

        # Pass through LSTM
        lstm_out, _ = self.lstm(x, (h0, c0)) # lstm_out shape: (batch, seq_len, lstm_hidden_size)

        # Use the output from the last time step for classification
        last_time_step_out = lstm_out[:, -1, :] # -> (batch, lstm_hidden_size)

        # Apply dropout and final linear layer
        out = self.dropout(last_time_step_out)
        logits = self.fc(out) # -> (batch, n_classes)

        # Squeeze the last dimension if n_classes is 1 (for BCEWithLogitsLoss)
        return logits.squeeze(-1) if self.n_classes == 1 else logits

# --- Training and Evaluation Functions ---
def train_epoch(model, dataloader, criterion, optimizer, device, grad_clip_norm):
    """ Trains model for one epoch, returns avg loss and avg grad norm. """
    model.train() # Set model to training mode
    total_loss = 0.0
    total_grad_norm = 0.0
    batches_processed = 0
    num_batches = len(dataloader)
    if num_batches == 0: logging.warning("Training dataloader empty."); return 0.0, 0.0

    for i, (sequences, labels) in enumerate(dataloader):
        sequences, labels = sequences.to(device), labels.to(device)

        optimizer.zero_grad(set_to_none=True) # More efficient zeroing

        # Forward pass
        outputs = model(sequences)
        loss = criterion(outputs, labels)

        # Check for NaN loss
        if torch.isnan(loss):
            logging.warning(f"NaN loss detected in training batch {i+1}. Skipping batch.")
            continue # Skip backward pass and optimizer step for this batch

        # Backward pass
        loss.backward()

        # Calculate gradient norm before clipping
        grad_norm = 0.0
        for p in model.parameters():
            if p.grad is not None:
                param_norm = p.grad.detach().data.norm(2)
                grad_norm += param_norm.item() ** 2
        grad_norm = grad_norm ** 0.5
        total_grad_norm += grad_norm

        # Gradient clipping (optional but recommended)
        if grad_clip_norm is not None and grad_clip_norm > 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=grad_clip_norm)

        # Optimizer step
        optimizer.step()

        total_loss += loss.item()
        batches_processed += 1

    # Calculate averages
    avg_loss = total_loss / batches_processed if batches_processed > 0 else 0.0
    avg_grad_norm = total_grad_norm / batches_processed if batches_processed > 0 else 0.0
    return avg_loss, avg_grad_norm

def evaluate(model, dataloader, criterion, device, return_preds=False):
    """ Evaluates the model, returns metrics and optionally preds/labels. """
    model.eval() # Set model to evaluation mode
    total_loss = 0.0
    all_preds_prob = []
    all_labels = []
    num_batches = len(dataloader)

    if num_batches == 0:
        logging.warning("Evaluation dataloader empty.")
        metrics = {'loss': float('nan'), 'f1': 0.0, 'acc': 0.0, 'auc': 0.5, 'gmean': 0.0}
        empty_preds = np.array([])
        return (metrics['loss'], metrics['f1'], metrics['acc'], metrics['auc'], metrics['gmean'], empty_preds, empty_preds) if return_preds else (metrics['loss'], metrics['f1'], metrics['acc'], metrics['auc'], metrics['gmean'])

    with torch.no_grad(): # Disable gradient calculations
        for sequences, labels in dataloader:
            sequences, labels = sequences.to(device), labels.to(device)

            # Forward pass
            outputs = model(sequences)
            # Use a dummy criterion if none provided (e.g., during prediction-only)
            loss = criterion(outputs, labels) if criterion else torch.tensor(0.0)

            if not torch.isnan(loss):
                total_loss += loss.item()
            else:
                logging.warning("NaN loss detected during evaluation.")

            # Get probabilities (apply sigmoid since using BCEWithLogitsLoss)
            probs = torch.sigmoid(outputs).cpu().numpy()
            all_preds_prob.extend(probs)
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / num_batches if num_batches > 0 and criterion else float('nan') # Loss is NaN if no criterion
    all_labels = np.array(all_labels)
    all_preds_prob = np.array(all_preds_prob)

    # Initialize metrics
    metrics = {'loss': avg_loss, 'f1': 0.0, 'acc': 0.0, 'auc': 0.5, 'gmean': 0.0}

    if len(all_labels) > 0 and len(all_preds_prob) > 0:
        # Ensure shapes are correct (squeeze if necessary)
        if all_preds_prob.ndim > 1 and all_preds_prob.shape[1] == 1: all_preds_prob = all_preds_prob.squeeze(-1)
        if all_labels.ndim > 1 and all_labels.shape[1] == 1: all_labels = all_labels.squeeze(-1)

        if len(all_labels) == len(all_preds_prob):
            try:
                # Calculate metrics using default 0.5 threshold first
                preds_binary_default = (all_preds_prob >= 0.5).astype(int)
                metrics['f1'] = f1_score(all_labels, preds_binary_default, zero_division=0)
                metrics['acc'] = accuracy_score(all_labels, preds_binary_default)

                # Calculate G-mean (requires recall for both classes)
                recall0 = recall_score(all_labels, preds_binary_default, pos_label=0, zero_division=0)
                recall1 = recall_score(all_labels, preds_binary_default, pos_label=1, zero_division=0)
                if recall0 >= 0 and recall1 >= 0: # Ensure valid recalls
                    metrics['gmean'] = math.sqrt(recall0 * recall1)
                else: metrics['gmean'] = 0.0 # Handle cases where recall is undefined

            except Exception as e:
                logging.error(f"Error calculating F1/Acc/G-mean metrics: {e}")

            try:
                # Calculate AUC (requires probabilities)
                if len(np.unique(all_labels)) > 1: # AUC is only defined for >1 class
                    metrics['auc'] = roc_auc_score(all_labels, all_preds_prob)
                else:
                    metrics['auc'] = 0.5 # Or float('nan')? Optuna prefers numbers.
                    logging.warning("Only one class present in evaluation labels. AUC set to 0.5.")
                    metrics['gmean'] = 0.0 # G-mean also requires both classes
            except ValueError as e:
                metrics['auc'] = 0.5 # Handle cases like all predictions being the same
                logging.error(f"AUC calculation ValueError: {e}. Setting AUC to 0.5.")
                metrics['gmean'] = 0.0
            except Exception as e:
                metrics['auc'] = 0.5
                logging.error(f"Unexpected error calculating AUC: {e}")
                metrics['gmean'] = 0.0
        else:
            logging.error(f"Label and prediction length mismatch during evaluation: {len(all_labels)} vs {len(all_preds_prob)}")
    else:
        logging.warning("No labels or predictions collected during evaluation.")

    if return_preds:
        return metrics['loss'], metrics['f1'], metrics['acc'], metrics['auc'], metrics['gmean'], all_labels, all_preds_prob
    else:
        return metrics['loss'], metrics['f1'], metrics['acc'], metrics['auc'], metrics['gmean']


# --- Optuna Objective Function ---
def objective(trial, config, train_dataset, val_dataset, n_features, y_train_raw):
    """ Optuna objective function for hyperparameter tuning. """
    params = {}
    search_space = config['OPTUNA_SEARCH_SPACE']
    device = config['DEVICE'] # Get device from config
    if not search_space: logging.error("Optuna search space missing."); raise optuna.exceptions.TrialPruned("OPTUNA_SEARCH_SPACE missing.")

    # --- Suggest Hyperparameters ---
    try:
        for name, definition in search_space.items():
            param_type = definition['type']
            args = definition.copy(); del args['type'] # Remove type key before passing to suggest_
            if param_type == 'categorical': params[name] = trial.suggest_categorical(name, **args)
            elif param_type == 'int': params[name] = trial.suggest_int(name, **args)
            elif param_type == 'float': params[name] = trial.suggest_float(name, **args)
            else: logging.warning(f"Unsupported Optuna parameter type '{param_type}' for '{name}'. Skipping.")
    except Exception as e:
        logging.error(f"Hyperparameter suggestion error in trial {trial.number}: {e}")
        raise optuna.exceptions.TrialPruned(f"HP suggestion failed: {e}") # Prune if suggestion fails

    # Extract suggested parameters
    # <<< CHANGE: Removed d_model and cnn_kernel_size >>>
    # d_model = params['d_model']; cnn_kernel_size = params['cnn_kernel_size'];
    lstm_hidden_size = params['lstm_hidden_size']
    lstm_n_layers = params['lstm_n_layers']; lstm_dropout = params['lstm_dropout']; fc_dropout = params['fc_dropout']
    lr = params['lr']; weight_decay = params['weight_decay']; batch_size = params['batch_size']
    use_dynamic_weighting = config['USE_DYNAMIC_WEIGHTING'] # Fixed setting from config
    param_str = ", ".join([f"{k}={v:.3g}" if isinstance(v, float) else f"{k}={v}" for k, v in params.items()]) # Format floats
    logging.debug(f"Trial {trial.number} Start: Params: {param_str}")

    # --- Model, Optimizer, Criterion Setup ---
    try:
        # <<< CHANGE: Instantiate LSTMGatedClassifier (or your chosen name) without CNN params >>>
        model = LSTMGatedClassifier(
            n_features=n_features,
            use_dynamic_weighting=use_dynamic_weighting, lstm_hidden_size=lstm_hidden_size,
            lstm_n_layers=lstm_n_layers, lstm_dropout=lstm_dropout, fc_dropout=fc_dropout,
            n_classes=1
        ).to(device)
        optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

        # Calculate positive class weight for loss function if enabled
        pos_weight_tensor = None
        if config['USE_WEIGHTED_LOSS']:
            neg_count = np.sum(y_train_raw == 0); pos_count = np.sum(y_train_raw == 1)
            if pos_count > 0 and neg_count > 0:
                # Calculate weight, clip to avoid extreme values
                pos_weight_val = np.clip(neg_count / pos_count, 1.0, 100.0)
                pos_weight_tensor = torch.tensor([pos_weight_val], device=device)
                logging.debug(f"Trial {trial.number}: Using weighted loss (pos_weight={pos_weight_val:.4f})")
                criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)
            else:
                logging.warning(f"Trial {trial.number}: Cannot calculate class weights (counts={neg_count},{pos_count}). Using unweighted loss.")
                criterion = nn.BCEWithLogitsLoss()
        else:
            criterion = nn.BCEWithLogitsLoss() # Default unweighted loss
    except Exception as model_init_e:
        logging.error(f"Trial {trial.number}: Model/Optimizer/Criterion setup failed: {model_init_e}")
        raise optuna.exceptions.TrialPruned(f"Setup failed: {model_init_e}")

    # --- DataLoader Setup ---
    try:
        # Use num_workers=0 for simplicity in typical notebook environments
        # pin_memory=True can speed up CPU->GPU transfer if using CUDA
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory=True, drop_last=True)
        val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=True, drop_last=False)
        if len(train_loader) == 0 or len(val_loader) == 0:
            logging.error(f"Trial {trial.number}: DataLoader creation resulted in empty loader(s).")
            raise optuna.exceptions.TrialPruned("DataLoader empty.")
    except Exception as e:
        logging.error(f"Trial {trial.number}: DataLoader creation failed: {e}")
        raise optuna.exceptions.TrialPruned(f"DataLoader failed: {e}")

    # --- Training Loop for Trial ---
    best_val_metric_in_trial = -float('inf') # Initialize for maximization
    metric_to_optimize = config['OPTUNA_METRIC_TO_OPTIMIZE']

    for epoch in range(config['N_EPOCHS_TUNING']):
        try:
            train_loss, avg_grad_norm = train_epoch(model, train_loader, criterion, optimizer, device, config['GRADIENT_CLIP_MAX_NORM'])
            val_loss, val_f1, val_acc, val_auc, val_gmean = evaluate(model, val_loader, criterion, device, return_preds=False)
            val_metrics_dict = {'loss': val_loss, 'f1': val_f1, 'acc': val_acc, 'auc': val_auc, 'gmean': val_gmean}

            # Check for NaN/inf values in metrics
            if any(map(lambda x: np.isnan(x) or np.isinf(x), [train_loss] + list(val_metrics_dict.values()))):
                logging.warning(f"Trial {trial.number} Epoch {epoch+1}: NaN/inf metric detected. Pruning trial.")
                raise optuna.exceptions.TrialPruned("NaN/inf metric.")

            current_val_metric = val_metrics_dict.get(metric_to_optimize)
            if current_val_metric is None or np.isnan(current_val_metric):
                logging.warning(f"Trial {trial.number} Epoch {epoch+1}: Target metric '{metric_to_optimize}' is NaN or None. Pruning trial.")
                raise optuna.exceptions.TrialPruned(f"Target metric '{metric_to_optimize}' was NaN/None.")

            # Log progress (especially if metric improves or it's the last epoch)
            is_last_epoch = epoch == config['N_EPOCHS_TUNING'] - 1
            if current_val_metric > best_val_metric_in_trial or is_last_epoch:
                logging.debug(f"Trial {trial.number} Ep {epoch+1}/{config['N_EPOCHS_TUNING']} | TrainLoss:{train_loss:.4f}|GradNorm:{avg_grad_norm:.4f}|Val {metric_to_optimize.upper()}:{current_val_metric:.4f}(Best:{max(best_val_metric_in_trial, current_val_metric):.4f})")

            # Report intermediate value to Optuna for pruning
            trial.report(current_val_metric, epoch)
            best_val_metric_in_trial = max(best_val_metric_in_trial, current_val_metric)

            # Check if the trial should be pruned based on intermediate results
            # Only prune after a minimum number of epochs
            if epoch + 1 >= config['MIN_EPOCH_OPTUNA'] and trial.should_prune():
                logging.debug(f"Trial {trial.number}: Pruned at epoch {epoch+1} by Optuna Pruner.")
                raise optuna.exceptions.TrialPruned(f"Pruned at epoch {epoch+1}")

            # Additional check for NaN/inf training loss (can happen mid-epoch)
            if np.isnan(train_loss) or np.isinf(train_loss):
                logging.warning(f"Trial {trial.number} Epoch {epoch+1}: Training loss became NaN/inf. Pruning trial.")
                raise optuna.exceptions.TrialPruned("NaN/inf training loss.")

        except optuna.exceptions.TrialPruned as pr_e:
            raise pr_e # Re-raise prune exceptions
        except Exception as e:
            logging.error(f"Trial {trial.number} Epoch {epoch+1}: Unexpected error during training/evaluation: {e}", exc_info=True)
            raise optuna.exceptions.TrialPruned(f"Error in epoch {epoch+1}: {e}") # Prune on other errors

    # Return the best validation metric achieved during the trial
    # Ensure a non-negative value is returned if metric can be negative but shouldn't be
    final_metric = max(0.0, best_val_metric_in_trial) if best_val_metric_in_trial > -float('inf') else 0.0
    logging.debug(f"Trial {trial.number} End: Best validation {metric_to_optimize.upper()}: {final_metric:.4f}")
    return final_metric

# --- Optuna Callback for detailed logging ---
def optuna_callback(study: optuna.study.Study, trial: optuna.trial.FrozenTrial):
    """ Callback function to log results after each Optuna trial. """
    log_level = logging.INFO # Use INFO level for standard trial logging
    if trial.state == TrialState.COMPLETE:
        metric_name = study.metric_names[0] if study.metric_names else "value" # Get metric name if available
        direction = study.direction.name
        value = trial.value
        best_trial = None
        try:
            # Filter only completed trials with valid values to find the best
            completed_trials = [t for t in study.trials if t.state == TrialState.COMPLETE and t.value is not None and not np.isnan(t.value)]
            if completed_trials: best_trial = study.best_trial # study.best_trial should handle direction
        except ValueError: # Can happen if no trials are complete yet
            best_trial = None

        current_best_value = best_trial.value if best_trial else (-float('inf') if study.direction == optuna.study.StudyDirection.MAXIMIZE else float('inf'))

        logging.log(log_level, f"Optuna Trial {trial.number} COMPLETED:")
        logging.log(log_level, f"  State: {trial.state}, Value ({metric_name} {direction}): {value:.5f}")
        param_str = ", ".join([f"{k}={v:.3g}" if isinstance(v, float) else f"{k}={v}" for k, v in trial.params.items()])
        logging.log(log_level, f"  Params: {param_str}")
        if best_trial:
            logging.log(log_level, f"  Current Best Trial: {best_trial.number}, Best Value: {current_best_value:.5f}")
            if trial.number == best_trial.number: logging.log(log_level, "  *** NEW BEST trial found! ***")
        else: logging.log(log_level, "  (No valid best trial yet)")

    elif trial.state == TrialState.PRUNED:
        logging.log(log_level, f"Optuna Trial {trial.number} PRUNED at step {trial.last_step}")
    elif trial.state == TrialState.FAIL:
        logging.log(logging.WARNING, f"Optuna Trial {trial.number} FAILED:")
        # Try to get the reason from system attributes if Optuna stored it
        fail_message = trial.system_attrs.get('fail_reason', 'Unknown reason')
        logging.log(logging.WARNING, f"  Reason: {fail_message}")


# --- Threshold Tuning ---
def find_optimal_threshold(y_true, y_prob, metric='f1'):
    """ Finds the optimal classification threshold based on a chosen metric on validation probabilities. """
    if y_true is None or y_prob is None or len(y_true)==0 or len(y_prob)==0:
        logging.warning("Empty input provided for threshold optimization. Returning default 0.5.")
        return 0.5
    # Ensure 1D numpy arrays
    y_true = np.array(y_true).astype(int).squeeze(); y_prob = np.array(y_prob).squeeze()
    if y_true.ndim > 1 or y_prob.ndim > 1:
        logging.error(f"Inputs must be 1D for threshold optimization. Got shapes {y_true.shape}, {y_prob.shape}. Returning 0.5.")
        return 0.5
    if len(y_true) != len(y_prob):
        logging.error(f"Length mismatch in threshold optimization: {len(y_true)} vs {len(y_prob)}. Returning 0.5.")
        return 0.5
    if len(np.unique(y_true)) < 2:
        logging.warning(f"Only one class found in y_true for threshold optimization. Returning 0.5.")
        return 0.5

    try:
        # Use precision_recall_curve to get candidate thresholds
        precision, recall, thresholds_pr = precision_recall_curve(y_true, y_prob)
        # Include 0.0, 0.5, 1.0 and unique thresholds from PR curve
        # Clip to avoid exact 0/1 which can cause issues with some metrics
        thresholds_to_check = np.unique(np.concatenate(([0.0, 0.5], thresholds_pr, [1.0])))
        thresholds_to_check = np.clip(thresholds_to_check, 1e-7, 1.0 - 1e-7)

        scores = []; recalls_0 = []; recalls_1 = []
        logging.debug(f"Optimizing threshold ({metric.upper()}). Checking {len(thresholds_to_check)} thresholds.")

        for t in thresholds_to_check:
            y_pred_t = (y_prob >= t).astype(int)
            score = 0.0
            # Calculate recalls needed for G-mean regardless of the target metric
            recall0 = recall_score(y_true, y_pred_t, pos_label=0, zero_division=0)
            recall1 = recall_score(y_true, y_pred_t, pos_label=1, zero_division=0)
            recalls_0.append(recall0); recalls_1.append(recall1)

            # Calculate the target score
            if metric == 'f1':
                score = f1_score(y_true, y_pred_t, pos_label=1, zero_division=0)
            elif metric == 'accuracy':
                score = accuracy_score(y_true, y_pred_t)
            elif metric == 'gmean':
                score = math.sqrt(recall0 * recall1) if recall0 >= 0 and recall1 >= 0 else 0.0
            else:
                logging.warning(f"Unsupported metric '{metric}' for threshold optimization. Defaulting to F1.")
                score = f1_score(y_true, y_pred_t, pos_label=1, zero_division=0)
            scores.append(score)

        if not scores:
            logging.warning("No scores calculated during threshold optimization. Returning 0.5.")
            return 0.5

        best_idx = np.argmax(scores)
        optimal_threshold = thresholds_to_check[best_idx]
        best_score = scores[best_idx]
        best_recall0 = recalls_0[best_idx]; best_recall1 = recalls_1[best_idx]
        logging.info(f"Optimal threshold search ({metric.upper()}): Best={optimal_threshold:.4f} (Score:{best_score:.4f}, R0:{best_recall0:.4f}, R1:{best_recall1:.4f})")

        # Return clipped threshold just in case
        return np.clip(optimal_threshold, 1e-7, 1.0 - 1e-7)

    except Exception as e:
        logging.error(f"Error during threshold optimization: {e}", exc_info=True)
        return 0.5


# --- Plotting and Results Saving ---
def save_static_training_plot(history, best_epoch, metric_name, save_path, config):
    """ Saves a static Matplotlib plot of the final training history. """
    if not history or not history.get('train_loss'):
        logging.warning("History dict empty or missing 'train_loss'. Skipping static plot save.")
        return

    fig, axes = plt.subplots(3, 1, figsize=(15, 10), sharex=True) # Keep size consistent
    fig.suptitle(f'Final Training History (Static - Best Val {metric_name.upper()} Epoch: {best_epoch if best_epoch else "N/A"})',
                 fontsize=config.get('PLOT_FONT_SIZE', 12) + 2)
    plt.style.use('seaborn-v0_8-whitegrid') # Use a seaborn style
    epochs_ran = range(1, len(history['train_loss']) + 1)
    font_size = config.get('PLOT_FONT_SIZE', 12)

    # --- Loss Plot (Top) ---
    ax1 = axes[0]
    ax1.plot(epochs_ran, history['train_loss'], label='Training Loss', marker='.', linestyle='-', color='royalblue')
    if history.get('val_loss'): ax1.plot(epochs_ran, history['val_loss'], label='Validation Loss', marker='.', linestyle='--', color='darkorange')
    if best_epoch: ax1.axvline(x=best_epoch, color='crimson', linestyle=':', linewidth=2, label=f"Best Epoch ({metric_name.upper()})")
    ax1.set_title('Model Loss'); ax1.set_ylabel('Loss'); ax1.legend(); ax1.tick_params(axis='y', labelsize=font_size-1)
    # Set Y limits dynamically based on plotted data
    all_losses = history.get('train_loss', []) + history.get('val_loss', [])
    valid_losses = [l for l in all_losses if l is not None and not np.isnan(l) and not np.isinf(l)]
    if valid_losses:
        min_loss=min(valid_losses); max_loss=max(valid_losses)
        y_bottom=max(0, min_loss*0.95-0.05) if min_loss>=0 else min_loss*1.05-0.05 # Add buffer, ensure >= 0 if possible
        y_top=max_loss*1.05+0.05 # Add buffer
        if y_top > y_bottom: ax1.set_ylim(bottom=y_bottom, top=y_top)
        else: ax1.set_ylim(bottom=y_bottom - 0.1, top=y_top + 0.1) # Handle flat lines
    else: ax1.set_ylim(bottom=0) # Default if no valid data

    # --- Metrics Plot (Middle) ---
    ax2 = axes[1]; plot_metrics = False
    if history.get('val_f1'): ax2.plot(epochs_ran, history['val_f1'], label='Val F1', color='darkorange', marker='.', linestyle='--'); plot_metrics=True
    if history.get('val_acc'): ax2.plot(epochs_ran, history['val_acc'], label='Val Acc', color='forestgreen', marker='.', linestyle=':'); plot_metrics=True
    if history.get('val_auc'): ax2.plot(epochs_ran, history['val_auc'], label='Val AUC', color='mediumpurple', marker='.', linestyle='-.'); plot_metrics=True
    if history.get('val_gmean'): ax2.plot(epochs_ran, history['val_gmean'], label='Val G-mean', color='gold', marker='^', linestyle=':'); plot_metrics=True
    if plot_metrics:
        if best_epoch: ax2.axvline(x=best_epoch, color='crimson', linestyle=':', linewidth=2, label=f'Best Epoch ({metric_name.upper()})')
        ax2.set_title('Validation Metrics'); ax2.set_ylabel('Score'); ax2.set_ylim(bottom=-0.05, top=1.05); ax2.legend(); ax2.tick_params(axis='y', labelsize=font_size-1)
    else: ax2.set_title('Validation Metrics (No Data)'); ax2.set_ylabel('Score'); ax2.set_ylim(bottom=-0.05, top=1.05)

    # --- Gradient Norm Plot (Bottom) ---
    ax3 = axes[2]
    if history.get('avg_grad_norm'):
        ax3.plot(epochs_ran, history['avg_grad_norm'], label='Avg Train Grad Norm', marker='.', linestyle='-', color='teal')
        ax3.set_title('Average Gradient Norm per Epoch'); ax3.set_ylabel('L2 Norm'); ax3.set_xlabel('Epochs'); ax3.legend()
        # Set Y limits dynamically
        valid_norms = [n for n in history['avg_grad_norm'] if n is not None and not np.isnan(n) and not np.isinf(n) and n > 0]
        if valid_norms:
            min_norm=min(valid_norms); max_norm=max(valid_norms)
            y_bottom_norm=min_norm*0.9; y_top_norm=max_norm*1.1
            if y_top_norm > y_bottom_norm: ax3.set_ylim(bottom=y_bottom_norm, top=y_top_norm)
            else: ax3.set_ylim(bottom=y_bottom_norm - 0.1, top=y_top_norm + 0.1) # Handle flat lines
        else: ax3.set_ylim(bottom=0) # Default if no valid data
    else: ax3.set_title('Gradient Norm (No Data)'); ax3.set_ylabel('L2 Norm'); ax3.set_xlabel('Epochs')
    ax3.tick_params(axis='both', which='major', labelsize=font_size-1)

    plt.tight_layout(rect=[0, 0.03, 1, 0.96]) # Adjust layout to prevent title overlap
    try:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        logging.info(f"Static training history plot saved: {save_path}")
    except Exception as e:
        logging.error(f"Error saving static training history plot: {e}")
    plt.close(fig) # Close the static figure to free memory

def plot_confusion_matrix(y_true, y_pred, save_path, config, title_suffix=""):
    """ Plots and saves a confusion matrix using seaborn. """
    if y_true is None or y_pred is None or len(y_true)==0 or len(y_pred)==0:
        logging.warning("Cannot plot confusion matrix: Input arrays are empty or None.")
        return
    try:
        y_true_int = np.array(y_true).astype(int); y_pred_int = np.array(y_pred).astype(int)
    except Exception as e:
        logging.error(f"Could not convert inputs to int for confusion matrix: {e}"); return
    if len(y_true_int) != len(y_pred_int):
        logging.error("Cannot plot confusion matrix: Input arrays have different lengths."); return

    try:
        cm = confusion_matrix(y_true_int, y_pred_int)
        class_names = ['Class 0', 'Class 1'] # Assuming binary classification
        plt.style.use('seaborn-v0_8-whitegrid') # Use a seaborn style
        plt.figure(figsize=config.get('PLOT_FIG_SIZE_CM', (8, 7)))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                    xticklabels=[f'Pred {n}' for n in class_names],
                    yticklabels=[f'Actual {n}' for n in class_names],
                    annot_kws={"size": config.get('PLOT_FONT_SIZE', 12)},
                    linewidths=.5, linecolor='lightgray')
        plt.title(f'Confusion Matrix{title_suffix}', fontsize=config.get('PLOT_FONT_SIZE', 12) + 1)
        plt.xlabel('Predicted Label'); plt.ylabel('True Label')
        plt.xticks(fontsize=config.get('PLOT_FONT_SIZE', 12)-1)
        plt.yticks(fontsize=config.get('PLOT_FONT_SIZE', 12)-1, rotation=0) # Keep y-axis labels horizontal
        plt.tight_layout(pad=1.5) # Add padding
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        logging.info(f"Confusion matrix plot saved: {save_path}")
    except Exception as e:
        logging.error(f"Error plotting/saving confusion matrix: {e}", exc_info=True)
    plt.close() # Close the static figure

def plot_actual_vs_prediction(pred_df, save_path, config):
    """ Plots actual vs predicted probabilities over the original index. """
    if pred_df is None or pred_df.empty:
        logging.warning("Prediction DataFrame is empty or None. Skipping actual vs prediction plot.")
        return
    if not all(col in pred_df.columns for col in ['actual', 'predicted_prob']):
        logging.warning(f"Prediction DataFrame missing required columns ('actual', 'predicted_prob'). Skipping plot. Columns: {pred_df.columns}")
        return

    try:
        plt.style.use('seaborn-v0_8-whitegrid')
        fig, ax = plt.subplots(figsize=config.get('PLOT_FIG_SIZE_ACT_PRED', (15, 7)))

        # Plot actual values (as markers or steps)
        ax.plot(pred_df.index, pred_df['actual'], label='Actual Target', marker='o', linestyle='None', color='black', markersize=4, alpha=0.7)

        # Plot predicted probabilities
        ax.plot(pred_df.index, pred_df['predicted_prob'], label='Predicted Probability', color='dodgerblue', linestyle='-', linewidth=1.5, alpha=0.8)

        # Optional: Add threshold line
        if 'optimal_threshold' in config:
            thresh = config['optimal_threshold']
            ax.axhline(y=thresh, color='red', linestyle='--', linewidth=1, label=f'Threshold ({thresh:.3f})')

        ax.set_title('Actual Target vs. Predicted Probability (Test Set)', fontsize=config.get('PLOT_FONT_SIZE', 12) + 1)
        ax.set_xlabel('Original Index / Time')
        ax.set_ylabel('Value / Probability')
        ax.legend()
        ax.set_ylim(-0.05, 1.05) # Probabilities are between 0 and 1

        # Format x-axis if the index is datetime-like
        if pd.api.types.is_datetime64_any_dtype(pred_df.index):
            fig.autofmt_xdate() # Auto-format dates
            # Optional: More specific formatting
            # ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d %H:%M'))
            # ax.xaxis.set_major_locator(mdates.AutoDateLocator())

        plt.tight_layout()
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        logging.info(f"Actual vs. Prediction plot saved: {save_path}")
        plt.close(fig) # Close the figure

    except Exception as e:
        logging.error(f"Error plotting/saving actual vs prediction plot: {e}", exc_info=True)
        plt.close() # Ensure figure is closed even on error

def save_results_summary(config, best_optuna_params, best_val_epoch_metrics, final_test_metrics, optimal_threshold, n_features, save_path, is_prediction_run=False):
    """ Saves a text summary of the configuration and results. """
    try:
        with open(save_path, 'w') as f:
            run_type = "Prediction" if is_prediction_run else "Training"
            f.write(f"==================== {run_type} Run Summary ====================\n")
            f.write(f"Run Timestamp: {config.get('RUN_TIMESTAMP_STR', 'N/A')}\n")
            f.write(f"Output Directory: {config.get('OUTPUT_SUBFOLDER_PATH', 'N/A')}\n")
            if is_prediction_run:
                f.write(f"Loaded Model Path: {config.get('LOAD_MODEL_PATH', 'N/A')}\n")
            f.write("-" * 90 + "\n")

            # --- Configuration (Only relevant parts for prediction, or full for training) ---
            if not is_prediction_run:
                f.write("--- Configuration Summary ---\n")
                keys_to_log = [
                    'SEED', 'DEVICE_STR', 'SEQUENCE_LENGTH', 'TRAIN_SPLIT_RATIO', 'VALIDATION_SPLIT_RATIO',
                    'USE_UNDERSAMPLING', 'USE_WEIGHTED_LOSS', 'USE_DYNAMIC_WEIGHTING',
                    'SKIP_OPTUNA_AND_USE_FIXED_PARAMS',
                    'FIXED_LSTM_HIDDEN_SIZE', 'FIXED_LSTM_LAYERS',
                    'FIXED_LSTM_DROPOUT', 'FIXED_FC_DROPOUT', 'FIXED_LR',
                    'FIXED_BATCH_SIZE', 'FIXED_WEIGHT_DECAY',
                    'DEFAULT_LSTM_HIDDEN_SIZE', 'DEFAULT_LSTM_LAYERS',
                    'DEFAULT_LSTM_DROPOUT', 'DEFAULT_FC_DROPOUT', 'DEFAULT_LEARNING_RATE',
                    'DEFAULT_BATCH_SIZE', 'DEFAULT_WEIGHT_DECAY',
                    'OPTIMIZE_THRESHOLD', 'THRESHOLD_OPTIMIZATION_METRIC',
                    'GRADIENT_CLIP_MAX_NORM', 'USE_LR_SCHEDULER', 'FINAL_N_EPOCHS', 'FINAL_EARLY_STOPPING_PATIENCE',
                    'MIN_EPOCH_FINAL', 'TUNE_HYPERPARAMETERS', 'OPTUNA_METRIC_TO_OPTIMIZE', 'N_OPTUNA_TRIALS',
                    'OPTUNA_TIMEOUT_SECONDS', 'OPTUNA_STUDY_NAME', 'OPTUNA_LOAD_EXISTING_STUDY',
                    'MIN_EPOCH_OPTUNA', 'N_EPOCHS_TUNING', 'OPTUNA_PRUNER_STARTUP_TRIALS',
                    'OPTUNA_PRUNER_WARMUP_STEPS', 'OPTUNA_PRUNER_INTERVAL_STEPS'
                ]
                if 'TICKER_PATTERN' in config: keys_to_log.insert(2, 'TICKER_PATTERN')
                skip_keys = ['BASE_OUTPUT_DIR', 'BEST_MODEL_FILENAME', 'TRAINING_HISTORY_PLOT_FILENAME',
                             'CONFUSION_MATRIX_PLOT_FILENAME', 'RESULTS_SUMMARY_FILENAME',
                             'PREDICTION_DATAFRAME_FILENAME', 'ACTUAL_VS_PREDICTION_PLOT_FILENAME',
                             'PREDICTION_CONFUSION_MATRIX_PLOT_FILENAME', 'PREDICTION_RESULTS_SUMMARY_FILENAME',
                             'PREDICTION_PREDICTION_DATAFRAME_FILENAME', 'PREDICTION_ACTUAL_VS_PREDICTION_PLOT_FILENAME',
                             'BEST_MODEL_SAVE_PATH', 'TRAINING_HISTORY_PLOT_PATH',
                             'CONFUSION_MATRIX_PLOT_PATH', 'RESULTS_SUMMARY_PATH',
                             'PREDICTION_DATAFRAME_PATH', 'ACTUAL_VS_PREDICTION_PLOT_PATH',
                             'OUTPUT_SUBFOLDER_PATH', 'RUN_TIMESTAMP_STR', 'LOAD_MODEL_PATH', 'DEVICE', # Skip derived paths and torch device object
                             'OPTUNA_SEARCH_SPACE', 'loaded_model_params'] # Skip complex objects
                for key in keys_to_log:
                    if key in config and key not in skip_keys:
                        f.write(f"{key}: {config[key]}\n")
                if config.get('TUNE_HYPERPARAMETERS', False) and not config.get('SKIP_OPTUNA_AND_USE_FIXED_PARAMS', False):
                    f.write("\n--- Optuna Search Space ---\n")
                    optuna_space = config.get('OPTUNA_SEARCH_SPACE', {})
                    for p_name, p_def in optuna_space.items(): f.write(f"  {p_name}: {p_def}\n")
                    f.write("--- End Optuna Search Space ---\n")
                f.write(f"N_FEATURES (Input Data): {n_features}\n"); f.write("-" * 90 + "\n")

                # Report which parameters were used for the final model
                f.write("\n" + "="*25 + " Final Hyperparameters Used " + "="*25 + "\n")
                if config.get('SKIP_OPTUNA_AND_USE_FIXED_PARAMS', False):
                    f.write("Used FIXED hyperparameters (Optuna skipped):\n")
                    # <<< CHANGE: Update fixed keys list >>>
                    fixed_keys = {
                        k.replace('FIXED_', '').lower(): f"FIXED_{k.replace('FIXED_', '')}"
                        for k in config if k.startswith('FIXED_') and 'CNN' not in k and 'D_MODEL' not in k
                    }
                    for param, conf_key in fixed_keys.items():
                        value = config.get(conf_key, 'N/A')
                        f.write(f"  {param}: {value:.4e}\n" if isinstance(value, float) and abs(value)<1e-2 else f"  {param}: {value}\n")
                elif config.get('TUNE_HYPERPARAMETERS', False) and best_optuna_params:
                    f.write("Used BEST Optuna hyperparameters:\n")
                    for param, value in best_optuna_params.items():
                        f.write(f"  {param}: {value:.4e}\n" if isinstance(value, float) and abs(value)<1e-2 else f"  {param}: {value}\n")
                    optuna_metric = config.get('OPTUNA_METRIC_TO_OPTIMIZE', 'N/A').upper()
                    best_optuna_val = best_val_epoch_metrics.get('best_optuna_value', float('nan'))
                    f.write(f"  (Best Optuna Metric Value ({optuna_metric}): {best_optuna_val:.4f})\n")
                else:
                    f.write("Used DEFAULT hyperparameters (Optuna disabled or failed):\n")
                    # <<< CHANGE: Update default keys list >>>
                    default_keys = {
                        k.replace('DEFAULT_', '').lower(): f"DEFAULT_{k.replace('DEFAULT_', '')}"
                        for k in config if k.startswith('DEFAULT_') and 'CNN' not in k and 'D_MODEL' not in k
                    }
                    for param, conf_key in default_keys.items():
                        value = config.get(conf_key, 'N/A')
                        f.write(f"  {param}: {value:.4e}\n" if isinstance(value, float) and abs(value)<1e-2 else f"  {param}: {value}\n")
                f.write("-" * 90 + "\n")

                # Best Validation Epoch Performance (Only for training run)
                f.write("\n" + "="*18 + " Best Validation Epoch Performance " + "="*18 + "\n")
                if best_val_epoch_metrics and 'epoch' in best_val_epoch_metrics:
                    metric_name = config.get('OPTUNA_METRIC_TO_OPTIMIZE', 'N/A').upper() # Metric used for early stopping
                    f.write(f"Best Epoch (>= {config.get('MIN_EPOCH_FINAL', 'N/A')} based on Validation '{metric_name}'): {best_val_epoch_metrics['epoch']}\n")
                    f.write(f"  Loss:   {best_val_epoch_metrics.get('loss', float('nan')):.4f}\n")
                    f.write(f"  F1:     {best_val_epoch_metrics.get('f1', float('nan')):.4f}\n")
                    f.write(f"  Acc:    {best_val_epoch_metrics.get('acc', float('nan')):.4f}\n")
                    f.write(f"  AUC:    {best_val_epoch_metrics.get('auc', float('nan')):.4f}\n")
                    f.write(f"  G-mean: {best_val_epoch_metrics.get('gmean', float('nan')):.4f}\n")
                else: f.write(f"No validation improvement recorded after epoch {config.get('MIN_EPOCH_FINAL', 'N/A')}.\n")
                f.write("-" * 90 + "\n")
            else: # Prediction run specific info
                f.write("--- Loaded Model Info ---\n")
                f.write(f"N_FEATURES (from loaded model): {n_features}\n")
                # Log loaded model parameters if available in config
                if 'loaded_model_params' in config:
                    f.write("Model Parameters (from loaded model):\n")
                    # <<< CHANGE: Print params from loaded config (already without CNN params) >>>
                    for param, value in config['loaded_model_params'].items():
                        f.write(f"  {param}: {value}\n")
                f.write("-" * 90 + "\n")


            # Final Model Performance (Test Set) - Applicable to both runs
            f.write("\n" + "="*25 + f" Final Model Performance (Test Set - {run_type} Run) " + "="*25 + "\n")
            thresh_metric_name = config.get('THRESHOLD_OPTIMIZATION_METRIC', 'N/A').upper() # Get from config if available
            if is_prediction_run:
                f.write(f"Threshold Used (Loaded from model file): {optimal_threshold:.4f}\n")
            else:
                f.write(f"Optimal Threshold (Found on Val Set, optimized for '{thresh_metric_name}'): {optimal_threshold:.4f}\n")

            if final_test_metrics:
                # Loss might be NaN if criterion wasn't used in prediction
                loss_val = final_test_metrics.get('loss', float('nan'))
                f.write(f"  Loss:   {loss_val:.4f}" + (" (N/A for prediction run)" if np.isnan(loss_val) and is_prediction_run else "") + "\n")
                f.write(f"  F1:     {final_test_metrics.get('f1', float('nan')):.4f}\n")
                f.write(f"  Acc:    {final_test_metrics.get('acc', float('nan')):.4f}\n")
                f.write(f"  AUC:    {final_test_metrics.get('auc', float('nan')):.4f}\n")
                f.write(f"  G-mean: {final_test_metrics.get('gmean', float('nan')):.4f}\n")
                f.write("\nClassification Report (Test Set):\n")
                f.write(final_test_metrics.get('report', "N/A") + "\n")
            else: f.write("Final evaluation on test set skipped or failed.\n")
            f.write("=" * 90 + "\n")

            # Saved Artifact Paths
            f.write("\n" + "="*35 + " Saved Artifact Paths " + "="*35 + "\n")
            if not is_prediction_run:
                f.write(f"Best Model:          {config.get('BEST_MODEL_SAVE_PATH', 'N/A')}\n")
                f.write(f"Predictions CSV:     {config.get('PREDICTION_DATAFRAME_PATH', 'N/A')}\n")
                f.write(f"Static History Plot: {config.get('TRAINING_HISTORY_PLOT_PATH', 'N/A')}\n")
                f.write(f"Confusion Matrix:    {config.get('CONFUSION_MATRIX_PLOT_PATH', 'N/A')}\n")
                f.write(f"Actual vs Pred Plot: {config.get('ACTUAL_VS_PREDICTION_PLOT_PATH', 'N/A')}\n")
            else:
                f.write(f"Predictions CSV:     {config.get('PREDICTION_DATAFRAME_PATH', 'N/A')}\n")
                f.write(f"Confusion Matrix:    {config.get('CONFUSION_MATRIX_PLOT_PATH', 'N/A')}\n")
                f.write(f"Actual vs Pred Plot: {config.get('ACTUAL_VS_PREDICTION_PLOT_PATH', 'N/A')}\n")
            f.write(f"Results Summary:     {save_path}\n") # This file path
            f.write("=" * 90 + "\n")

        logging.info(f"Results summary saved successfully to: {save_path}")
    except IOError as e:
        logging.error(f"Error writing results summary file: {e}")
    except Exception as e:
        logging.error(f"Unexpected error saving results summary: {e}", exc_info=True)


# ========================================================
# Main Execution Logic (Training Pipeline)
# ========================================================
def run_training_pipeline(data_df, config):
    """ Orchestrates the entire training and evaluation pipeline with live monitoring and final plot display. """
    pipeline_start_time = time.time()
    # <<< CHANGE: Updated log message >>>
    logging.info(f"--- Starting LSTM-Gated Training & Evaluation Pipeline (v{config.get('VERSION', 'unknown')}) ---")
    set_seed(config['SEED'])
    logging.info(f"Using Device: {config['DEVICE_STR']}") # Log the string representation

    # --- Create Timestamped Output Subfolder ---
    try:
        now = datetime.now(); timestamp_str = now.strftime("runtime%Y_%m_%d_%H_%M_%S")
        output_subfolder_path = os.path.join(config.get('BASE_OUTPUT_DIR', 'models'), timestamp_str)
        os.makedirs(output_subfolder_path, exist_ok=True)
        logging.info(f"Created output subfolder: {output_subfolder_path}")
        # Create a run-specific config copy and add derived paths
        run_config = config.copy()
        run_config['OUTPUT_SUBFOLDER_PATH'] = output_subfolder_path
        run_config['RUN_TIMESTAMP_STR'] = timestamp_str
        run_config['BEST_MODEL_SAVE_PATH'] = os.path.join(output_subfolder_path, config.get('BEST_MODEL_FILENAME', 'best_model.pth'))
        run_config['TRAINING_HISTORY_PLOT_PATH'] = os.path.join(output_subfolder_path, config.get('TRAINING_HISTORY_PLOT_FILENAME', 'training_history_STATIC.png'))
        run_config['CONFUSION_MATRIX_PLOT_PATH'] = os.path.join(output_subfolder_path, config.get('CONFUSION_MATRIX_PLOT_FILENAME', 'confusion_matrix.png'))
        run_config['RESULTS_SUMMARY_PATH'] = os.path.join(output_subfolder_path, config.get('RESULTS_SUMMARY_FILENAME', 'results_summary.txt'))
        run_config['PREDICTION_DATAFRAME_PATH'] = os.path.join(output_subfolder_path, config.get('PREDICTION_DATAFRAME_FILENAME', 'test_predictions.csv'))
        run_config['ACTUAL_VS_PREDICTION_PLOT_PATH'] = os.path.join(output_subfolder_path, config.get('ACTUAL_VS_PREDICTION_PLOT_FILENAME', 'actual_vs_prediction.png'))
    except Exception as path_e:
        logging.critical(f"Failed to create output directory '{output_subfolder_path}': {path_e}"); return None # Return None on failure

    # --- 1. Data Preparation ---
    test_indices = None ; scaler_obj = None
    try:
        X_train, y_train, X_val, y_val, X_test, y_test, n_features, scaler_obj, y_train_raw, test_indices = split_apply_undersample_scale(data_df, run_config)
        logging.info(f"Data preparation successful. n_features: {n_features}")
        if not all(isinstance(arr, np.ndarray) for arr in [X_train, y_train, X_val, y_val, X_test, y_test]):
            raise TypeError("Data preparation did not return numpy arrays.")
        # Allow empty test set if splits result in it
        if X_test.size == 0 and len(y_test) > 0:
            logging.warning("Test features (X_test) are empty, but test labels (y_test) exist. Check split ratios and data size.")
        if test_indices is None and len(y_test) > 0:
            logging.warning("Test indices could not be aligned. Prediction DataFrame and plot might be unavailable or incorrect.")
    except Exception as e:
        logging.critical(f"Data preparation failed: {e}", exc_info=True); return None

    # --- 2. Dataset Creation ---
    try:
        train_dataset = TimeSeriesDataset(X_train, y_train)
        val_dataset = TimeSeriesDataset(X_val, y_val)
        # Handle potentially empty test set
        test_dataset = TimeSeriesDataset(X_test, y_test) if X_test.size > 0 else None
        logging.info(f"Datasets created. Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset) if test_dataset else 0}")
        if not all(len(ds) > 0 for ds in [train_dataset, val_dataset]): # Only require train/val to be non-empty
            raise ValueError("Empty Train or Validation PyTorch Dataset created.")
    except Exception as e:
        logging.critical(f"Dataset creation failed: {e}", exc_info=True); return None

    # --- 3. Hyperparameter Tuning (Optuna) ---
    # (Keep Optuna logic as is, but uses the modified objective function)
    best_optuna_params = {}; best_optuna_value = None
    study_storage = None

    if run_config.get('SKIP_OPTUNA_AND_USE_FIXED_PARAMS', False):
        logging.info("\n--- Skipping Hyperparameter Tuning (Optuna) - Using FIXED parameters ---")
        best_optuna_params = {}
    elif run_config.get('TUNE_HYPERPARAMETERS', False):
        logging.info(f"\n--- Starting Hyperparameter Tuning (Optuna) ---")
        logging.info(f"Optimizing: {run_config.get('OPTUNA_METRIC_TO_OPTIMIZE', 'gmean').upper()}")
        # <<< CHANGE: Updated default study name >>>
        study_storage = f"sqlite:///{run_config.get('BASE_OUTPUT_DIR', 'models')}/optuna_studies.db"
        study_name = run_config.get('OPTUNA_STUDY_NAME', 'lstm_gated_tuning_default')
        load_existing = run_config.get('OPTUNA_LOAD_EXISTING_STUDY', True)

        try:
            os.makedirs(os.path.dirname(study_storage.replace("sqlite:///", "")), exist_ok=True)
            if not load_existing:
                logging.info(f"Attempting to start a fresh Optuna study '{study_name}'. Deleting existing study if found...")
                try:
                    optuna.delete_study(study_name=study_name, storage=study_storage)
                    logging.info(f"Deleted existing study '{study_name}'.")
                except KeyError:
                    logging.info(f"No existing study named '{study_name}' found to delete.")
                except Exception as del_e:
                    logging.warning(f"Could not delete existing study '{study_name}': {del_e}. Proceeding to create study.")

            study = optuna.create_study(
                study_name=study_name, storage=study_storage, direction="maximize",
                sampler=optuna.samplers.TPESampler(seed=run_config['SEED']),
                pruner=optuna.pruners.MedianPruner(
                    n_startup_trials=run_config.get('OPTUNA_PRUNER_STARTUP_TRIALS', 5),
                    n_warmup_steps=run_config.get('OPTUNA_PRUNER_WARMUP_STEPS', 10),
                    interval_steps=run_config.get('OPTUNA_PRUNER_INTERVAL_STEPS', 3)
                ), load_if_exists=load_existing
            )

            if load_existing and len(study.trials) > 0: logging.info(f"Loaded existing Optuna study '{study_name}' with {len(study.trials)} previous trials.")
            elif not load_existing: logging.info(f"Created a new Optuna study '{study_name}'.")
            else: logging.info(f"Created a new Optuna study '{study_name}' (no existing study found to load).")

            start_time_optuna = time.time()
            objective_with_args = partial(objective, config=run_config, train_dataset=train_dataset, val_dataset=val_dataset, n_features=n_features, y_train_raw=y_train_raw)
            study.optimize(objective_with_args,
                           n_trials=run_config.get('N_OPTUNA_TRIALS', 30),
                           timeout=run_config.get('OPTUNA_TIMEOUT_SECONDS', None),
                           callbacks=[optuna_callback], catch=(Exception,))
            optuna_duration = time.time() - start_time_optuna
            logging.info(f"Optuna tuning finished in {optuna_duration:.2f} seconds.")

            logging.info("Retrieving best hyperparameters from Optuna study...")
            completed_trials = [t for t in study.trials if t.state == TrialState.COMPLETE and t.value is not None and not np.isnan(t.value)]
            if completed_trials:
                best_trial = study.best_trial
                best_optuna_value = best_trial.value
                best_optuna_params = best_trial.params
                logging.info(f"Best Optuna trial: #{best_trial.number}, Value ({run_config.get('OPTUNA_METRIC_TO_OPTIMIZE','N/A').upper()}): {best_optuna_value:.5f}")
                logging.info(f"Best Optuna Params: {best_optuna_params}")
            else:
                logging.warning("No valid Optuna trials completed. Will use default hyperparameters.")
                best_optuna_params = {}

        except DuplicatedStudyError:
            logging.error(f"Optuna study '{study_name}' already exists, but load_if_exists=False was likely used without successful deletion.")
            best_optuna_params = {}
        except Exception as e:
            logging.error(f"An error occurred during Optuna hyperparameter tuning: {e}", exc_info=True)
            best_optuna_params = {}
    else:
        logging.info("\n--- Hyperparameter tuning (Optuna) is disabled - Using DEFAULT parameters ---")
        best_optuna_params = {}


    # --- 4. Determine Final Hyperparameters ---
    # (Keep logic as is, but uses the modified default/fixed keys)
    final_params = {}
    if run_config.get('SKIP_OPTUNA_AND_USE_FIXED_PARAMS', False):
        logging.info("Using FIXED hyperparameters for final training.")
        # <<< CHANGE: Update fixed keys >>>
        fixed_keys = {
            k.replace('FIXED_', '').lower(): f"FIXED_{k.replace('FIXED_', '')}"
            for k in run_config if k.startswith('FIXED_') and 'CNN' not in k and 'D_MODEL' not in k
        }
        for param_key, config_key in fixed_keys.items(): final_params[param_key] = run_config.get(config_key)
    elif run_config.get('TUNE_HYPERPARAMETERS', False) and best_optuna_params:
        logging.info("Using BEST Optuna hyperparameters for final training.")
        final_params = best_optuna_params
    else:
        source = "DEFAULT (Optuna failed/no valid trials)" if run_config.get('TUNE_HYPERPARAMETERS', False) else "DEFAULT (Optuna disabled)"
        logging.info(f"Using {source} hyperparameters.")
        # <<< CHANGE: Update default keys >>>
        default_keys = {
            k.replace('DEFAULT_', '').lower(): f"DEFAULT_{k.replace('DEFAULT_', '')}"
            for k in run_config if k.startswith('DEFAULT_') and 'CNN' not in k and 'D_MODEL' not in k
        }
        for param_key, config_key in default_keys.items(): final_params[param_key] = run_config.get(config_key)

    # <<< CHANGE: Remove CNN-specific defaults >>>
    # final_params.setdefault('d_model', run_config.get('DEFAULT_D_MODEL', 64))
    # final_params.setdefault('cnn_kernel_size', run_config.get('DEFAULT_CNN_KERNEL_SIZE', 3))
    final_params.setdefault('lstm_hidden_size', run_config.get('DEFAULT_LSTM_HIDDEN_SIZE', 64))
    final_params.setdefault('lstm_n_layers', run_config.get('DEFAULT_LSTM_LAYERS', 1))
    final_params.setdefault('lstm_dropout', run_config.get('DEFAULT_LSTM_DROPOUT', 0.2))
    final_params.setdefault('fc_dropout', run_config.get('DEFAULT_FC_DROPOUT', 0.3))
    final_params.setdefault('lr', run_config.get('DEFAULT_LEARNING_RATE', 5e-4))
    final_params.setdefault('batch_size', run_config.get('DEFAULT_BATCH_SIZE', 128))
    final_params.setdefault('weight_decay', run_config.get('DEFAULT_WEIGHT_DECAY', 5e-4))

    if None in final_params.values():
        logging.critical(f"One or more final hyperparameters are None: {final_params}. Cannot proceed.")
        return None

    # <<< CHANGE: Update model_init_params to exclude CNN params >>>
    model_init_params = {
        'n_features': n_features,
        # 'd_model': final_params['d_model'],             # Removed
        # 'cnn_kernel_size': final_params['cnn_kernel_size'], # Removed
        'use_dynamic_weighting': run_config['USE_DYNAMIC_WEIGHTING'],
        'lstm_hidden_size': final_params['lstm_hidden_size'],
        'lstm_n_layers': final_params['lstm_n_layers'],
        'lstm_dropout': final_params['lstm_dropout'],
        'fc_dropout': final_params['fc_dropout'],
        'n_classes': 1
    }
    final_batch_size = final_params['batch_size']
    final_lr = final_params['lr']
    final_weight_decay = final_params['weight_decay']

    logging.info("\n--- Final Model Configuration & Hyperparameters ---")
    logging.info(f"  Input Features: {n_features}")
    param_source = "FIXED" if run_config.get('SKIP_OPTUNA_AND_USE_FIXED_PARAMS', False) else \
                     ("Optuna (Best Trial)" if run_config.get('TUNE_HYPERPARAMETERS', False) and best_optuna_params else "DEFAULT")
    logging.info(f"  Parameter Source: {param_source}")
    # <<< CHANGE: Logging uses the modified model_init_params >>>
    for k, v in model_init_params.items(): logging.info(f"  Model Init Param - {k}: {v}")
    logging.info(f"  Training - Batch Size: {final_batch_size}, LR: {final_lr:.2e}, Weight Decay: {final_weight_decay:.2e}")


    # --- 5. Final Model Training ---
    logging.info("\n--- Starting Final Model Training ---")
    final_model=None; final_optimizer=None; final_criterion=None; final_scheduler=None
    history = {'train_loss': [], 'val_loss': [], 'val_f1': [], 'val_acc': [], 'val_auc': [], 'val_gmean': [], 'avg_grad_norm': []}
    best_val_metric_post_min_epoch = -float('inf')
    best_epoch_num_post_min_epoch = 0
    best_epoch_metrics_post_min_epoch = {}
    epochs_no_improve_post_min_epoch = 0
    model_saved_flag = False
    metric_to_monitor = run_config.get('OPTUNA_METRIC_TO_OPTIMIZE', 'gmean')
    optimal_threshold = 0.5 # Initialize optimal threshold
    device = run_config['DEVICE'] # Get device from config

    # --- Initialize Live Plot ---
    live_fig = None
    if ipython_display_available: # Only try if display is available
        try:
            live_fig = go.FigureWidget(make_subplots(rows=3, cols=1, shared_xaxes=True,
                                                     subplot_titles=("Loss", "Validation Metrics", "Avg Gradient Norm")))
            # Add traces...
            live_fig.add_trace(go.Scatter(name='Training Loss', mode='lines+markers', line=dict(color='royalblue')), row=1, col=1)
            live_fig.add_trace(go.Scatter(name='Validation Loss', mode='lines+markers', line=dict(color='darkorange', dash='dash')), row=1, col=1)
            live_fig.add_trace(go.Scatter(name='Val F1', mode='lines+markers', line=dict(color='darkorange', dash='dash')), row=2, col=1)
            live_fig.add_trace(go.Scatter(name='Val Acc', mode='lines+markers', line=dict(color='forestgreen', dash='dot')), row=2, col=1)
            live_fig.add_trace(go.Scatter(name='Val AUC', mode='lines+markers', line=dict(color='mediumpurple', dash='dashdot')), row=2, col=1)
            live_fig.add_trace(go.Scatter(name='Val G-mean', mode='lines+markers', line=dict(color='gold', dash='longdash')), row=2, col=1)
            live_fig.add_trace(go.Scatter(name='Avg Grad Norm', mode='lines+markers', line=dict(color='teal')), row=3, col=1)
            live_fig.update_layout(
                title=f'Live Training Progress (Monitoring: {metric_to_monitor.upper()})',
                height=900, # Keep increased height
                hovermode="x unified",
                xaxis3_title='Epoch',
                yaxis_title='Loss',
                yaxis2_title='Score',
                yaxis3_title='L2 Norm',
                yaxis2_range=[-0.05, 1.05],
                legend_traceorder="reversed",
                shapes=()
            )
            display(live_fig) # Call display here
            logging.info("Live plot initialized.")
        except Exception as plot_init_e:
            logging.error(f"Failed to initialize or display live Plotly plot: {plot_init_e}. Live plotting disabled.", exc_info=True)
            live_fig = None
    else:
        logging.info("Live plotting disabled as IPython.display is not available.")


    # --- Final Training Setup ---
    try:
        final_train_loader = DataLoader(train_dataset, batch_size=final_batch_size, shuffle=True, num_workers=0, pin_memory=True, drop_last=True)
        final_val_loader = DataLoader(val_dataset, batch_size=final_batch_size, shuffle=False, num_workers=0, pin_memory=True, drop_last=False)
        if len(final_train_loader) == 0 or len(final_val_loader) == 0: raise ValueError("Empty final DataLoader(s).")

        # <<< CHANGE: Instantiate LSTMGatedClassifier using updated params >>>
        final_model = LSTMGatedClassifier(**model_init_params).to(device)
        logging.info("\n--- Final Model Architecture ---"); logging.info(final_model)
        total_params = sum(p.numel() for p in final_model.parameters() if p.requires_grad)
        logging.info(f"Total Trainable Parameters: {total_params:,}")

        final_optimizer = optim.AdamW(final_model.parameters(), lr=final_lr, weight_decay=final_weight_decay)

        pos_weight_tensor = None
        if run_config['USE_WEIGHTED_LOSS']:
            neg_count = np.sum(y_train_raw == 0); pos_count = np.sum(y_train_raw == 1)
            if pos_count > 0 and neg_count > 0:
                pos_weight_val = np.clip(neg_count / pos_count, 1.0, 100.0)
                pos_weight_tensor = torch.tensor([pos_weight_val], device=device)
                logging.info(f"Final Training: Using weighted loss (pos_weight={pos_weight_val:.4f})")
                final_criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)
            else:
                logging.warning("Final Training: Cannot calculate class weights. Using unweighted loss.")
                final_criterion = nn.BCEWithLogitsLoss()
        else:
            logging.info("Final Training: Using unweighted loss.")
            final_criterion = nn.BCEWithLogitsLoss()

        if run_config.get('USE_LR_SCHEDULER', False):
            final_scheduler = ReduceLROnPlateau(
                final_optimizer, mode='max',
                factor=run_config.get('LR_SCHEDULER_FACTOR', 0.1),
                patience=run_config.get('LR_SCHEDULER_PATIENCE', 15),
                min_lr=run_config.get('LR_SCHEDULER_MIN_LR', 1e-5)
                # Keep verbose=False removed if desired
            )
        else: final_scheduler = None

        logging.info(f"\n--- Starting final training loop ---")
        logging.info(f"Max Epochs: {run_config['FINAL_N_EPOCHS']}")
        logging.info(f"Min Epoch Save/Stop: {run_config['MIN_EPOCH_FINAL']}")
        logging.info(f"Patience: {run_config['FINAL_EARLY_STOPPING_PATIENCE']}")
        logging.info(f"Monitoring Metric for Save/Stop/LR Schedule: {metric_to_monitor.upper()}")

        start_time_final_train = time.time()
        # --- Final Training Loop ---
        for epoch in range(run_config['FINAL_N_EPOCHS']):
            epoch_start_time = time.time()
            train_loss, avg_grad_norm = train_epoch(final_model, final_train_loader, final_criterion, final_optimizer, device, run_config['GRADIENT_CLIP_MAX_NORM'])
            val_loss, val_f1, val_acc, val_auc, val_gmean, y_true_val, y_prob_val = evaluate(final_model, final_val_loader, final_criterion, device, return_preds=True)
            val_metrics_dict = {'loss': val_loss, 'f1': val_f1, 'acc': val_acc, 'auc': val_auc, 'gmean': val_gmean}
            epoch_duration = time.time() - epoch_start_time

            if any(map(lambda x: np.isnan(x) or np.isinf(x), [train_loss, avg_grad_norm] + list(val_metrics_dict.values()))):
                logging.error(f"Epoch {epoch+1}: NaN/inf detected in metrics. Stopping training.")
                break

            history['train_loss'].append(train_loss); history['val_loss'].append(val_loss)
            history['val_f1'].append(val_f1); history['val_acc'].append(val_acc)
            history['val_auc'].append(val_auc); history['val_gmean'].append(val_gmean)
            history['avg_grad_norm'].append(avg_grad_norm)

            current_lr = final_optimizer.param_groups[0]['lr']
            log_msg = (f"Epoch {epoch+1}/{run_config['FINAL_N_EPOCHS']} | Time:{epoch_duration:.2f}s | LR:{current_lr:.2e} | "
                       f"Loss(Trn/Val): {train_loss:.4f}/{val_loss:.4f} | GradNorm:{avg_grad_norm:.4f} | "
                       f"F1:{val_f1:.4f}|Acc:{val_acc:.4f}|AUC:{val_auc:.4f}|G-mean:{val_gmean:.4f}")
            logging.info(log_msg)

            # Update live plot
            if live_fig:
                try:
                    with live_fig.batch_update():
                        epochs_so_far = list(range(1, epoch + 2))
                        live_fig.data[0].x = epochs_so_far; live_fig.data[0].y = history['train_loss']
                        live_fig.data[1].x = epochs_so_far; live_fig.data[1].y = history['val_loss']
                        live_fig.data[2].x = epochs_so_far; live_fig.data[2].y = history['val_f1']
                        live_fig.data[3].x = epochs_so_far; live_fig.data[3].y = history['val_acc']
                        live_fig.data[4].x = epochs_so_far; live_fig.data[4].y = history['val_auc']
                        live_fig.data[5].x = epochs_so_far; live_fig.data[5].y = history['val_gmean']
                        live_fig.data[6].x = epochs_so_far; live_fig.data[6].y = history['avg_grad_norm']

                        if best_epoch_num_post_min_epoch > 0:
                            current_shapes = list(live_fig.layout.shapes)
                            # Keep change: Access shape name via attribute and check existence
                            filtered_shapes = [s for s in current_shapes if not (hasattr(s, 'name') and s.name == 'best_epoch_vline')]
                            vline_shape = dict(type="line", x0=best_epoch_num_post_min_epoch, y0=0, x1=best_epoch_num_post_min_epoch, y1=1, xref="x", yref="paper", line=dict(color="red", width=2, dash="dash"), name='best_epoch_vline')
                            live_fig.layout.shapes = tuple(filtered_shapes + [vline_shape])
                except Exception as plot_update_e:
                    # Log warning instead of error to avoid interrupting training for plot issues
                    logging.warning(f"Error updating live plot: {plot_update_e}", exc_info=True)


            # Check for Improvement / Early Stopping / Model Saving
            current_val_metric = val_metrics_dict.get(metric_to_monitor)
            if current_val_metric is None or np.isnan(current_val_metric):
                logging.warning(f"Epoch {epoch+1}: Monitored metric '{metric_to_monitor}' is NaN/None. Skipping improvement check.")
                continue

            if final_scheduler:
                old_lr = final_optimizer.param_groups[0]['lr']
                final_scheduler.step(current_val_metric)
                new_lr = final_optimizer.param_groups[0]['lr']
                if new_lr < old_lr: logging.info(f"Epoch {epoch+1}: Reducing LR via scheduler to {new_lr:.2e}")

            if epoch + 1 >= run_config['MIN_EPOCH_FINAL']:
                if current_val_metric > best_val_metric_post_min_epoch:
                    logging.info(f"  => Epoch {epoch+1}: Validation {metric_to_monitor.upper()} IMPROVED: {best_val_metric_post_min_epoch:.5f} -> {current_val_metric:.5f}.")
                    best_val_metric_post_min_epoch = current_val_metric
                    best_epoch_num_post_min_epoch = epoch + 1
                    best_epoch_metrics_post_min_epoch = {**val_metrics_dict, 'epoch': best_epoch_num_post_min_epoch}
                    if best_optuna_value is not None: best_epoch_metrics_post_min_epoch['best_optuna_value'] = best_optuna_value
                    epochs_no_improve_post_min_epoch = 0

                    if run_config.get('OPTIMIZE_THRESHOLD', True) and len(y_true_val) > 0:
                        thresh_opt_metric = run_config.get('THRESHOLD_OPTIMIZATION_METRIC', 'gmean')
                        logging.debug(f"Finding optimal threshold on current validation predictions (optimizing for '{thresh_opt_metric.upper()}')...")
                        optimal_threshold = find_optimal_threshold(y_true_val, y_prob_val, metric=thresh_opt_metric)
                    else:
                        optimal_threshold = 0.5

                    # <<< CHANGE: Checkpoint saves the modified model_init_params >>>
                    checkpoint = {
                        'epoch': best_epoch_num_post_min_epoch,
                        'model_state_dict': final_model.state_dict(),
                        'model_init_params': model_init_params, # This now holds LSTM-only params
                        'n_features': n_features,
                        'optimal_threshold': optimal_threshold,
                        'best_val_metric': best_val_metric_post_min_epoch,
                        'monitoring_metric': metric_to_monitor
                    }
                    try:
                        torch.save(checkpoint, run_config['BEST_MODEL_SAVE_PATH'])
                        logging.info(f"  => Saved best model checkpoint to {run_config['BEST_MODEL_SAVE_PATH']} (Threshold: {optimal_threshold:.4f})")
                        model_saved_flag = True
                    except Exception as save_e:
                        logging.error(f"  => FAILED to save model checkpoint: {save_e}")
                        model_saved_flag = False
                else:
                    epochs_no_improve_post_min_epoch += 1
                    if epochs_no_improve_post_min_epoch % 5 == 0 or epochs_no_improve_post_min_epoch >= run_config['FINAL_EARLY_STOPPING_PATIENCE']:
                        best_info = f"Best was {best_val_metric_post_min_epoch:.5f} (Epoch {best_epoch_num_post_min_epoch})" if best_epoch_num_post_min_epoch > 0 else "(No best epoch recorded yet)"
                        logging.info(f"  => Epoch {epoch+1}: Val {metric_to_monitor.upper()} ({current_val_metric:.5f}) hasn't improved for {epochs_no_improve_post_min_epoch} epochs. {best_info}.")

                    if epochs_no_improve_post_min_epoch >= run_config['FINAL_EARLY_STOPPING_PATIENCE']:
                        logging.info(f"\nEarly stopping triggered after {epoch + 1} epochs.")
                        break
            else:
                logging.debug(f"  (Epoch {epoch+1} < Min Epoch {run_config['MIN_EPOCH_FINAL']}. Improvement checks deferred.)")
        # --- End of Training Loop ---

        final_train_duration = time.time() - start_time_final_train
        logging.info(f"\nFinal training loop finished in {final_train_duration:.2f} seconds.")

        logging.info("Saving final static training history plot...")
        save_static_training_plot(history, best_epoch_num_post_min_epoch, metric_to_monitor, run_config['TRAINING_HISTORY_PLOT_PATH'], run_config)

        if best_epoch_num_post_min_epoch > 0:
            logging.info(f"Best validation {metric_to_monitor.upper()} achieved after epoch {run_config['MIN_EPOCH_FINAL']}: {best_val_metric_post_min_epoch:.5f} (at Epoch {best_epoch_num_post_min_epoch})")
        else:
            logging.warning(f"No improvement in validation {metric_to_monitor.upper()} was observed after epoch {run_config['MIN_EPOCH_FINAL']}.")

    except Exception as setup_e:
        logging.critical(f"Fatal error during final training setup or loop: {setup_e}", exc_info=True)
        model_saved_flag = False

    # --- 6. Final Evaluation on Test Set ---
    # (Keep logic as is - loading mechanism handles modified model_init_params)
    logging.info("\n--- Final Evaluation on TEST Set (Training Run) ---")
    final_test_metrics = None; prediction_df = None
    model_file_path = run_config['BEST_MODEL_SAVE_PATH']
    model_file_exists = os.path.exists(model_file_path)

    loaded_optimal_threshold = 0.5
    eval_model = None # Initialize eval_model to None

    if model_saved_flag and model_file_exists:
        try:
            checkpoint = torch.load(model_file_path, map_location=device, weights_only=False)
            logging.info(f"Checkpoint loaded successfully from {model_file_path}")
            loaded_optimal_threshold = checkpoint.get('optimal_threshold', 0.5) # Use .get for safety
            logging.info(f"Using optimal threshold loaded from checkpoint: {loaded_optimal_threshold:.4f}")

            # <<< CHANGE: This will load the modified params without CNN info >>>
            eval_model_params = checkpoint.get('model_init_params')
            if eval_model_params:
                # <<< CHANGE: Instantiates LSTMGatedClassifier using loaded params >>>
                eval_model = LSTMGatedClassifier(**eval_model_params).to(device)
                eval_model.load_state_dict(checkpoint['model_state_dict'])
                eval_model.eval()
                logging.info("Best model instantiated and state loaded successfully for evaluation.")
            else:
                logging.error("Model init params not found in checkpoint. Cannot evaluate.")
                model_saved_flag = False # Indicate evaluation cannot proceed

        except Exception as load_e:
            logging.error(f"Failed to load model checkpoint for evaluation: {load_e}", exc_info=True)
            model_saved_flag = False
    elif not model_saved_flag:
        logging.warning("Skipping final evaluation as best model was not saved.")
    elif not model_file_exists:
        logging.error(f"Best model file not found at expected path: {model_file_path}. Skipping final evaluation.")

    if eval_model and test_dataset: # Check eval_model is successfully loaded
        try:
            logging.info(f"Evaluating final model on Test set using threshold: {loaded_optimal_threshold:.4f}...")
            test_loader = DataLoader(test_dataset, batch_size=final_batch_size, shuffle=False, num_workers=0, pin_memory=True)
            criterion_for_eval = final_criterion if 'final_criterion' in locals() and final_criterion else None
            if not criterion_for_eval: logging.warning("Evaluating test set without calculating loss (criterion unavailable).")

            test_loss, _, _, test_auc, _, y_true_test, y_prob_test = evaluate(eval_model, test_loader, criterion_for_eval, device, return_preds=True)

            if y_true_test is not None and y_prob_test is not None and len(y_true_test) > 0:
                y_pred_test_optimal = (y_prob_test >= loaded_optimal_threshold).astype(int)
                test_f1 = f1_score(y_true_test, y_pred_test_optimal, zero_division=0)
                test_acc = accuracy_score(y_true_test, y_pred_test_optimal)
                r0 = recall_score(y_true_test, y_pred_test_optimal, pos_label=0, zero_division=0)
                r1 = recall_score(y_true_test, y_pred_test_optimal, pos_label=1, zero_division=0)
                test_gmean = math.sqrt(r0 * r1) if r0 >= 0 and r1 >= 0 else 0.0
                try:
                    test_report = classification_report(y_true_test, y_pred_test_optimal, target_names=['Class 0', 'Class 1'], zero_division=0, digits=4)
                except Exception as report_err:
                    test_report = f"Classification report generation failed: {report_err}"
                    logging.error(test_report)

                final_test_metrics = {'loss':test_loss,'f1':test_f1,'acc':test_acc,'auc':test_auc,'gmean':test_gmean,'report':test_report}
                logging.info(f"\n--- Final Test Set Performance ---")
                logging.info(f"(Using Threshold: {loaded_optimal_threshold:.4f})")
                logging.info(f"Loss:{test_loss:.4f} | Acc:{test_acc:.4f} | F1:{test_f1:.4f} | AUC:{test_auc:.4f} | G-mean:{test_gmean:.4f}")
                logging.info("\nClassification Report (Test Set):\n" + test_report)

                if test_indices is not None and len(test_indices) == len(y_true_test):
                    logging.info("Creating prediction DataFrame...")
                    prediction_df = pd.DataFrame({
                        'actual': y_true_test.astype(int),
                        'predicted_prob': y_prob_test,
                        'predicted_class': y_pred_test_optimal
                    }, index=test_indices)
                    pred_save_path = run_config['PREDICTION_DATAFRAME_PATH']
                    try:
                        prediction_df.to_csv(pred_save_path)
                        logging.info(f"Test predictions DataFrame saved to: {pred_save_path}")
                    except Exception as df_save_e:
                        logging.error(f"Failed to save prediction DataFrame: {df_save_e}")
                else:
                    logging.warning("Cannot create prediction DataFrame: Test indices are missing, None, or length mismatch.")
                    prediction_df = None

                plot_confusion_matrix(y_true_test, y_pred_test_optimal, run_config['CONFUSION_MATRIX_PLOT_PATH'], run_config,
                                      title_suffix=f" (Test, Thresh={loaded_optimal_threshold:.2f})")

                if prediction_df is not None:
                    run_config['optimal_threshold'] = loaded_optimal_threshold
                    plot_actual_vs_prediction(prediction_df, run_config['ACTUAL_VS_PREDICTION_PLOT_PATH'], run_config)
                else:
                    logging.warning("Skipping Actual vs Prediction plot because prediction DataFrame could not be created.")
            else:
                logging.error("Test set evaluation returned empty predictions or labels.")
        except Exception as final_eval_e:
            logging.error(f"An error occurred during final model evaluation: {final_eval_e}", exc_info=True)
            final_test_metrics = None
    elif not test_dataset:
        logging.warning("Skipping final evaluation as the test dataset is empty.")
    # else: Model loading failed or wasn't saved, message already logged

    # --- 7. Save Results Summary ---
    logging.info("Saving final results summary...")
    save_results_summary(run_config, best_optuna_params, best_epoch_metrics_post_min_epoch, final_test_metrics, loaded_optimal_threshold, n_features, run_config['RESULTS_SUMMARY_PATH'], is_prediction_run=False)

    # --- 8. Display Saved Plots (if in suitable environment) ---
    # (Keep logic as is)
    if ipython_display_available:
        logging.info("\n--- Displaying Saved Plots (if possible) ---")
        try:
            static_plot_path = run_config.get('TRAINING_HISTORY_PLOT_PATH')
            cm_plot_path = run_config.get('CONFUSION_MATRIX_PLOT_PATH')
            act_pred_plot_path = run_config.get('ACTUAL_VS_PREDICTION_PLOT_PATH')

            displayed_plot = False
            if static_plot_path and os.path.exists(static_plot_path):
                logging.info(f"Displaying static training history plot: {static_plot_path}")
                display(Image(filename=static_plot_path)); displayed_plot = True
            else: logging.warning("Static training history plot not found or path not configured.")

            displayed_cm = False
            if cm_plot_path and os.path.exists(cm_plot_path):
                logging.info(f"Displaying confusion matrix plot: {cm_plot_path}")
                display(Image(filename=cm_plot_path)); displayed_cm = True
            else: logging.warning("Confusion matrix plot not found or path not configured.")

            displayed_act_pred = False
            if act_pred_plot_path and os.path.exists(act_pred_plot_path):
                logging.info(f"Displaying actual vs prediction plot: {act_pred_plot_path}")
                display(Image(filename=act_pred_plot_path)); displayed_act_pred = True
            else: logging.warning("Actual vs prediction plot not found or path not configured.")

            if not displayed_plot and not displayed_cm and not displayed_act_pred:
                logging.info("No plots were displayed inline. Check the saved files.")

        except Exception as display_e:
            logging.error(f"An error occurred while trying to display plots: {display_e}", exc_info=True)
    else:
        logging.info("\n--- Skipping plot display (IPython.display not available) ---")

    # --- Pipeline End ---
    pipeline_duration = time.time() - pipeline_start_time
    # <<< CHANGE: Updated log message >>>
    logging.info(f"\n--- LSTM-Gated Training Pipeline Finished ---")
    logging.info(f"Total execution time: {pipeline_duration:.2f} seconds.")
    logging.info(f"All results saved in: {run_config['OUTPUT_SUBFOLDER_PATH']}")
    return run_config['OUTPUT_SUBFOLDER_PATH']


# ========================================================
# Prediction Pipeline Function
# ========================================================
def run_prediction_pipeline(data_df, model_path, base_config):
    """
    Loads a trained model, regenerates the exact same test data split,
    runs predictions, and saves evaluation results.
    """
    pipeline_start_time = time.time()
    logging.info(f"\n--- Starting Prediction Pipeline (Using Model: {model_path}) ---")

    if not os.path.exists(model_path):
        logging.critical(f"Model file not found: {model_path}"); return

    # --- Load Model and Configuration ---
    try:
        # Set weights_only=False to load the entire checkpoint object
        # This is generally safe if the checkpoint comes from a trusted source (like this script)
        # but be cautious loading checkpoints from unknown origins.
        device = base_config.get('DEVICE') # Get device from config
        checkpoint = torch.load(model_path, map_location=device, weights_only=False)
        logging.info(f"Checkpoint loaded successfully from {model_path}")
        model_state_dict = checkpoint['model_state_dict']
        # <<< CHANGE: Loads the modified model_init_params (without CNN) >>>
        model_init_params = checkpoint['model_init_params']
        n_features_loaded = checkpoint['n_features']
        optimal_threshold_loaded = checkpoint['optimal_threshold']
        # Use the SEED from the base config for reproducibility during data split
        seed_loaded = base_config['SEED']
        logging.info(f"Loaded model checkpoint. Features: {n_features_loaded}, Threshold: {optimal_threshold_loaded:.4f}, Seed: {seed_loaded}")
    except Exception as e:
        logging.critical(f"Failed to load checkpoint from {model_path}: {e}", exc_info=True); return

    # --- Create Timestamped Output Subfolder for Predictions ---
    try:
        now = datetime.now(); timestamp_str = now.strftime("prediction_runtime%Y_%m_%d_%H_%M_%S")
        # Place prediction output in a subfolder within the base output dir
        pred_output_subfolder_path = os.path.join(base_config.get('BASE_OUTPUT_DIR', 'models'), timestamp_str)
        os.makedirs(pred_output_subfolder_path, exist_ok=True)
        logging.info(f"Created prediction output subfolder: {pred_output_subfolder_path}")

        # Create a config for this prediction run, inheriting from base_config
        pred_run_config = base_config.copy()
        pred_run_config['OUTPUT_SUBFOLDER_PATH'] = pred_output_subfolder_path
        pred_run_config['RUN_TIMESTAMP_STR'] = timestamp_str
        pred_run_config['LOAD_MODEL_PATH'] = model_path
        # <<< CHANGE: Stores the modified loaded_model_params >>>
        pred_run_config['loaded_model_params'] = model_init_params # Store loaded params for summary
        # Define paths for prediction artifacts using specific filenames from base_config
        # <<< CHANGE: Updated default filenames (optional) >>>
        pred_run_config['CONFUSION_MATRIX_PLOT_PATH'] = os.path.join(pred_output_subfolder_path, base_config.get('PREDICTION_CONFUSION_MATRIX_PLOT_FILENAME', 'RELOADED_lstm_confusion_matrix.png'))
        pred_run_config['RESULTS_SUMMARY_PATH'] = os.path.join(pred_output_subfolder_path, base_config.get('PREDICTION_RESULTS_SUMMARY_FILENAME', 'RELOADED_lstm_results_summary.txt'))
        pred_run_config['PREDICTION_DATAFRAME_PATH'] = os.path.join(pred_output_subfolder_path, base_config.get('PREDICTION_PREDICTION_DATAFRAME_FILENAME', 'RELOADED_lstm_test_predictions.csv'))
        pred_run_config['ACTUAL_VS_PREDICTION_PLOT_PATH'] = os.path.join(pred_output_subfolder_path, base_config.get('PREDICTION_ACTUAL_VS_PREDICTION_PLOT_FILENAME', 'RELOADED_lstm_actual_vs_prediction.png'))
        # Add loaded threshold to config for plotting/summary
        pred_run_config['optimal_threshold'] = optimal_threshold_loaded

    except Exception as path_e:
        logging.critical(f"Failed to create prediction output directory '{pred_output_subfolder_path}': {path_e}"); return

    # --- Regenerate Data Split and Scale (using loaded SEED and config) ---
    test_indices = None; scaler_obj = None
    try:
        # Ensure the prediction config uses the loaded SEED for the split
        pred_run_config['SEED'] = seed_loaded
        logging.info(f"Regenerating data split using SEED: {pred_run_config['SEED']}")

        # Pass the prediction config to the splitting function
        _, _, _, _, X_test, y_test, n_features_regen, scaler_obj, _, test_indices = split_apply_undersample_scale(data_df, pred_run_config)

        if n_features_regen != n_features_loaded:
            logging.warning(f"Number of features in regenerated data ({n_features_regen}) does not match loaded model ({n_features_loaded}). Check data source.")
            # Continue for now, but this might indicate an issue

        if X_test.size == 0 and len(y_test) > 0:
            logging.warning("Regenerated test features (X_test) are empty, but test labels (y_test) exist.")
        if test_indices is None and len(y_test) > 0:
            logging.warning("Regenerated test indices could not be aligned.")

        logging.info(f"Data regeneration successful. Test set shape: {X_test.shape}")

    except Exception as e:
        logging.critical(f"Data regeneration failed during prediction run: {e}", exc_info=True); return

    # --- Instantiate Model and Load State ---
    try:
        # <<< CHANGE: Use the modified LSTMGatedClassifier class >>>
        # Use the parameters saved in the checkpoint
        model = LSTMGatedClassifier(**model_init_params).to(device)
        model.load_state_dict(model_state_dict)
        model.eval() # Set to evaluation mode
        logging.info("Model instantiated and state loaded successfully.")
    except Exception as e:
        logging.critical(f"Failed to instantiate or load model state: {e}", exc_info=True); return

    # --- Create Test Dataset and DataLoader ---
    if X_test.size > 0 and len(y_test) > 0:
        try:
            test_dataset = TimeSeriesDataset(X_test, y_test)
            # Determine batch size (can use default or load from config if needed)
            pred_batch_size = pred_run_config['DEFAULT_BATCH_SIZE']
            test_loader = DataLoader(test_dataset, batch_size=pred_batch_size, shuffle=False, num_workers=0, pin_memory=True)
            logging.info(f"Test DataLoader created with batch size: {pred_batch_size}")
        except Exception as e:
            logging.critical(f"Failed to create test dataset/loader: {e}", exc_info=True); return
    else:
        logging.warning("Test data (X_test or y_test) is empty. Skipping prediction.")
        test_loader = None

    # --- Run Prediction and Evaluation ---
    final_test_metrics = None; prediction_df = None
    if model and test_loader:
        try:
            logging.info(f"Running predictions on regenerated Test set using loaded threshold: {optimal_threshold_loaded:.4f}...")
            # Evaluate without criterion to only get predictions
            _, _, _, test_auc, _, y_true_test, y_prob_test = evaluate(model, test_loader, None, device, return_preds=True) # Pass None for criterion

            if y_true_test is not None and y_prob_test is not None and len(y_true_test) > 0:
                y_pred_test_optimal = (y_prob_test >= optimal_threshold_loaded).astype(int)

                # Calculate metrics
                test_f1 = f1_score(y_true_test, y_pred_test_optimal, zero_division=0)
                test_acc = accuracy_score(y_true_test, y_pred_test_optimal)
                r0 = recall_score(y_true_test, y_pred_test_optimal, pos_label=0, zero_division=0)
                r1 = recall_score(y_true_test, y_pred_test_optimal, pos_label=1, zero_division=0)
                test_gmean = math.sqrt(r0 * r1) if r0 >= 0 and r1 >= 0 else 0.0
                try:
                    test_report = classification_report(y_true_test, y_pred_test_optimal, target_names=['Class 0', 'Class 1'], zero_division=0, digits=4)
                except Exception as report_err:
                    test_report = f"Classification report generation failed: {report_err}"
                    logging.error(test_report)

                # Store metrics (loss is NaN as no criterion was used)
                final_test_metrics = {'loss':float('nan'),'f1':test_f1,'acc':test_acc,'auc':test_auc,'gmean':test_gmean,'report':test_report}

                logging.info(f"\n--- Prediction Run: Test Set Performance ---")
                logging.info(f"(Using Threshold: {optimal_threshold_loaded:.4f})")
                logging.info(f"Acc:{test_acc:.4f} | F1:{test_f1:.4f} | AUC:{test_auc:.4f} | G-mean:{test_gmean:.4f}")
                logging.info("\nClassification Report (Test Set):\n" + test_report)

                # Create and Save Prediction DataFrame
                if test_indices is not None and len(test_indices) == len(y_true_test):
                    logging.info("Creating prediction DataFrame...")
                    prediction_df = pd.DataFrame({
                        'actual': y_true_test.astype(int),
                        'predicted_prob': y_prob_test,
                        'predicted_class': y_pred_test_optimal
                    }, index=test_indices)
                    pred_df_save_path = pred_run_config['PREDICTION_DATAFRAME_PATH']
                    try:
                        prediction_df.to_csv(pred_df_save_path)
                        logging.info(f"Test predictions DataFrame saved to: {pred_df_save_path}")
                    except Exception as df_save_e:
                        logging.error(f"Failed to save prediction DataFrame: {df_save_e}")
                else:
                    logging.warning("Cannot create prediction DataFrame: Test indices missing or length mismatch.")
                    prediction_df = None

                # Plot Confusion Matrix
                cm_save_path = pred_run_config['CONFUSION_MATRIX_PLOT_PATH']
                plot_confusion_matrix(y_true_test, y_pred_test_optimal, cm_save_path, pred_run_config,
                                      title_suffix=f" (RELOADED Test, Thresh={optimal_threshold_loaded:.2f})")

                # Plot Actual vs Prediction
                if prediction_df is not None:
                    act_pred_save_path = pred_run_config['ACTUAL_VS_PREDICTION_PLOT_PATH']
                    plot_actual_vs_prediction(prediction_df, act_pred_save_path, pred_run_config) # optimal_threshold is already in pred_run_config
                else:
                    logging.warning("Skipping Actual vs Prediction plot (prediction DataFrame unavailable).")
            else:
                logging.error("Prediction run evaluation returned empty predictions or labels.")
        except Exception as pred_eval_e:
            logging.error(f"An error occurred during prediction run evaluation: {pred_eval_e}", exc_info=True)
            final_test_metrics = None

    # --- Save Prediction Results Summary ---
    logging.info("Saving prediction results summary...")
    summary_save_path = pred_run_config['RESULTS_SUMMARY_PATH']
    save_results_summary(pred_run_config, None, None, final_test_metrics, optimal_threshold_loaded, n_features_loaded, summary_save_path, is_prediction_run=True) # Pass None for Optuna/Val metrics

    # --- Display Plots ---
    if ipython_display_available:
        logging.info("\n--- Displaying Saved Prediction Plots (if possible) ---")
        try:
            cm_plot_path = pred_run_config.get('CONFUSION_MATRIX_PLOT_PATH')
            act_pred_plot_path = pred_run_config.get('ACTUAL_VS_PREDICTION_PLOT_PATH')

            displayed_cm = False
            if cm_plot_path and os.path.exists(cm_plot_path):
                logging.info(f"Displaying reloaded confusion matrix plot: {cm_plot_path}")
                display(Image(filename=cm_plot_path)); displayed_cm = True
            else: logging.warning("Reloaded confusion matrix plot not found or path not configured.")

            displayed_act_pred = False
            if act_pred_plot_path and os.path.exists(act_pred_plot_path):
                logging.info(f"Displaying reloaded actual vs prediction plot: {act_pred_plot_path}")
                display(Image(filename=act_pred_plot_path)); displayed_act_pred = True
            else: logging.warning("Reloaded actual vs prediction plot not found or path not configured.")

            if not displayed_cm and not displayed_act_pred:
                logging.info("No prediction plots were displayed inline. Check the saved files.")

        except Exception as display_e:
            logging.error(f"An error occurred while trying to display prediction plots: {display_e}", exc_info=True)
    else:
        logging.info("\n--- Skipping plot display (IPython.display not available) ---")


    # --- Pipeline End ---
    pipeline_duration = time.time() - pipeline_start_time
    logging.info(f"\n--- Prediction Pipeline Finished ---")
    logging.info(f"Total execution time: {pipeline_duration:.2f} seconds.")
    logging.info(f"Prediction results saved in: {pred_run_config['OUTPUT_SUBFOLDER_PATH']}")


# ========================================================
# Standalone Execution Block
# ========================================================
if __name__ == "__main__":

    # ========================================================
    # Configuration Parameters (Moved from Global Scope)
    # ========================================================
    config_run = {
        # --- General Settings ---
        'SEED': 42,
#         'DEVICE_STR': "cuda" if torch.cuda.is_available() else "cpu", # Store string representation
        'DEVICE_STR': "cpu",

        # --- Data Preprocessing (Post-Loading) ---
        'SEQUENCE_LENGTH': 5,
        'TRAIN_SPLIT_RATIO': 0.70,
        'VALIDATION_SPLIT_RATIO': 0.15,

        # Data Balancing
        'USE_UNDERSAMPLING': False, # Undersampling disabled
        'USE_WEIGHTED_LOSS': True, # Enable weighted loss for imbalance

        # --- Model Architecture (Fixed & Defaults) ---
        'USE_DYNAMIC_WEIGHTING': False, # Gating mechanism
        'USE_DUMMY_SCALER':False,
        # --- Option to skip Optuna and use fixed parameters ---
        'SKIP_OPTUNA_AND_USE_FIXED_PARAMS': True, # Set True to bypass Optuna

        'FIXED_LSTM_HIDDEN_SIZE': 16,
        'FIXED_LSTM_LAYERS': 1,
        'FIXED_LSTM_DROPOUT': 0.15,
        'FIXED_FC_DROPOUT': 0.15,
        'FIXED_LR': 1e-03,
        'FIXED_BATCH_SIZE': 250,
        'FIXED_WEIGHT_DECAY': 1e-04,

#         'DEFAULT_LSTM_HIDDEN_SIZE': 32,
#         'DEFAULT_LSTM_LAYERS': 2,
#         'DEFAULT_LSTM_DROPOUT': 0.2,
#         'DEFAULT_FC_DROPOUT': 0.3,
#         'DEFAULT_LEARNING_RATE': 5e-4,
#         'DEFAULT_BATCH_SIZE': 128,
#         'DEFAULT_WEIGHT_DECAY': 5e-4,

        # --- Training & Optimization ---
        'OPTIMIZE_THRESHOLD': True,
        'THRESHOLD_OPTIMIZATION_METRIC': 'gmean', # Options: 'f1', 'accuracy', 'gmean'
        'TUNE_HYPERPARAMETERS': False, # Master switch for enabling Optuna (ignored if SKIP_OPTUNA... is True)
        'GRADIENT_CLIP_MAX_NORM': 5.0,
        'USE_LR_SCHEDULER': False,
        'LR_SCHEDULER_FACTOR': 0.005,
        'LR_SCHEDULER_PATIENCE': 25,
        'LR_SCHEDULER_MIN_LR': 1e-5,
        'FINAL_N_EPOCHS': 500,
        'FINAL_EARLY_STOPPING_PATIENCE': 500,
        'MIN_EPOCH_FINAL': 500, # Min epochs before saving best model / early stopping in final training

        # --- Optuna Configuration (only used if TUNE_HYPERPARAMETERS=True AND SKIP_OPTUNA...=False) ---
        'N_OPTUNA_TRIALS': 30,
        'OPTUNA_TIMEOUT_SECONDS': 1800, # Max time for Optuna study
        # <<< CHANGE: Updated default study name >>>
        'OPTUNA_STUDY_NAME': "lstm_gated_tuning_refactored", # Updated study name
        'OPTUNA_LOAD_EXISTING_STUDY': False, # Set to False to force a fresh study
        'OPTUNA_METRIC_TO_OPTIMIZE': "gmean", # Metric Optuna maximizes during trials
        'OPTUNA_PRUNER_STARTUP_TRIALS': 5,
        'OPTUNA_PRUNER_WARMUP_STEPS': 10,
        'OPTUNA_PRUNER_INTERVAL_STEPS': 3,
        'N_EPOCHS_TUNING': 80, # Max epochs per Optuna trial
        'MIN_EPOCH_OPTUNA': 8,

        'OPTUNA_SEARCH_SPACE': {
            # 'd_model':            {'type': 'categorical', 'choices': [32, 64, 128]}, # Removed
            # 'cnn_kernel_size':    {'type': 'categorical', 'choices': [3, 5]},       # Removed
            'lstm_hidden_size':   {'type': 'categorical', 'choices': [32, 64, 96, 128]},
            'lstm_n_layers':      {'type': 'int', 'low': 1, 'high': 2},
            'lstm_dropout':       {'type': 'float', 'low': 0.1, 'high': 0.4},
            'fc_dropout':         {'type': 'float', 'low': 0.15, 'high': 0.5},
            'lr':                 {'type': 'float', 'low': 1e-4, 'high': 5e-3, 'log': True},
            'weight_decay':       {'type': 'float', 'low': 1e-5, 'high': 1e-2, 'log': True},
            'batch_size':         {'type': 'categorical', 'choices': [64, 128, 256]}
        },

        # --- Output & Logging ---
        # LOG_LEVEL and LOG_FORMAT are handled by basicConfig at the top
        'BASE_OUTPUT_DIR': 'models_refactored_lstm_only', # <<< CHANGE: Updated output dir name (optional) >>>
        # <<< CHANGE: Updated default filenames (optional) >>>
        # Filenames for training run
        'BEST_MODEL_FILENAME': 'best_lstm_gated_model.pth',
        'TRAINING_HISTORY_PLOT_FILENAME': "training_history_lstm_gated_STATIC.png",
        'CONFUSION_MATRIX_PLOT_FILENAME': "confusion_matrix_lstm_gated.png",
        'RESULTS_SUMMARY_FILENAME': "final_results_summary_lstm_gated.txt",
        'PREDICTION_DATAFRAME_FILENAME': "test_predictions_lstm_gated.csv",
        'ACTUAL_VS_PREDICTION_PLOT_FILENAME': "actual_vs_prediction_plot_lstm_gated.png",
        # Filenames for prediction-only run
        'PREDICTION_CONFUSION_MATRIX_PLOT_FILENAME': "RELOADED_confusion_matrix_lstm_gated.png",
        'PREDICTION_RESULTS_SUMMARY_FILENAME': "RELOADED_results_summary_lstm_gated.txt",
        'PREDICTION_PREDICTION_DATAFRAME_FILENAME': "RELOADED_test_predictions_lstm_gated.csv",
        'PREDICTION_ACTUAL_VS_PREDICTION_PLOT_FILENAME': "RELOADED_actual_vs_prediction_plot_lstm_gated.png",

        # --- Plotting ---
        'PLOT_FONT_SIZE': 12,
        'PLOT_FIG_SIZE_CM': (8, 7), # Used for confusion matrix
        'PLOT_FIG_SIZE_ACT_PRED': (15, 7), # Size for the actual vs prediction plot

        # --- Script Version ---
        'VERSION': '1.1.0_lstm_only' # <<< CHANGE: Updated version (optional) >>>
    }
    # Add the actual torch device object to the config
    config_run['DEVICE'] = torch.device(config_run['DEVICE_STR'])

    training_output_dir = run_training_pipeline(data_df, config_run)

    # ========================================================
    # Run Prediction Pipeline (using the model just trained)
    # ========================================================
    if training_output_dir and os.path.exists(os.path.join(training_output_dir, config_run['BEST_MODEL_FILENAME'])):
        logging.info("\n" + "="*20 + " Running Prediction Pipeline " + "="*20)
        model_to_load_path = os.path.join(training_output_dir, config_run['BEST_MODEL_FILENAME'])
        # Pass the same config_run used for training, as it contains necessary parameters like SEED, SEQUENCE_LENGTH etc.
        # The prediction function will use these to regenerate the correct test split.
        run_prediction_pipeline(data_df, model_to_load_path, config_run)
    else:
        logging.warning("\nSkipping prediction pipeline because training failed or model file was not found.")

